# 05 — Finishing Position and Non-Finish Outcomes

## Bounded question

How are finishing positions and non-finish outcomes represented in the source, and which raw and candidate result attributes can be derived reliably without concealing anomalies or source conventions?

## Source and analytical grain

This notebook uses the immutable SQLite source:

- Database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- Table: `data`
- Data-row predicate: `rowid <> 1`

The primary analytical grain is the source runner record, interpreted within the provisional race identity of `date + course + off`.

## Method

The notebook will first profile the observed representation and interaction of:

- `pos`
- `ran`
- `btn`
- `ovr_btn`

It will then investigate numeric finishing positions, non-finish codes, dead heats, disqualifications, amended outcomes, sentinel values, and contradictory or apparently impossible records.

Exact raw values will be preserved throughout. Candidate mappings will only be proposed after the source conventions and anomalies have been observed directly.

Final target-schema design remains deferred.

In [3]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"

connection = sqlite3.connect(f"file:{SOURCE_DB}?mode=ro", uri=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source database: {SOURCE_DB}")
print(f"Read-only connection established: {SOURCE_DB.exists()}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Data-row predicate: {DATA_ROW_PREDICATE}")

Project root: /home/rob/Documents/inside-rails-horse-racing
Source database: /home/rob/Documents/inside-rails-horse-racing/data/raw/form_2015-present/form_2015-present/raceform.db
Read-only connection established: True
Source table: data
Data-row predicate: rowid <> 1


In [4]:
source_check = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS data_rows,
        COUNT(pos) AS non_null_pos,
        COUNT(ran) AS non_null_ran,
        COUNT(btn) AS non_null_btn,
        COUNT(ovr_btn) AS non_null_ovr_btn
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

source_check

,data_rows,non_null_pos,non_null_ran,non_null_btn,non_null_ovr_btn
0,1851285,1851285,1851285,1851285,1851285


In [5]:
result_field_storage = pd.read_sql_query(
    f"""
    SELECT
        typeof(pos) AS pos_storage,
        typeof(ran) AS ran_storage,
        typeof(btn) AS btn_storage,
        typeof(ovr_btn) AS ovr_btn_storage,
        COUNT(*) AS rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    GROUP BY
        typeof(pos),
        typeof(ran),
        typeof(btn),
        typeof(ovr_btn)
    ORDER BY rows DESC
    """,
    connection,
)

result_field_storage

,pos_storage,ran_storage,btn_storage,ovr_btn_storage,rows
0,integer,integer,real,real,859833
1,integer,integer,integer,integer,353745
2,integer,integer,integer,real,307170
3,integer,integer,real,integer,235926
4,text,integer,text,text,93992
5,text,integer,integer,integer,239
6,text,integer,real,real,217
7,text,integer,integer,real,96
8,text,integer,real,integer,67


In [6]:
text_pos_values = pd.read_sql_query(
    f"""
    SELECT
        TRIM(CAST(pos AS TEXT)) AS raw_pos,
        COUNT(*) AS rows,
        COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
    GROUP BY TRIM(CAST(pos AS TEXT))
    ORDER BY rows DESC, raw_pos
    """,
    connection,
)

print(f"Distinct textual pos values: {len(text_pos_values):,}")
text_pos_values

Distinct textual pos values: 11


,raw_pos,rows,provisional_races
0,PU,65832,33433
1,F,15681,12920
2,UR,9527,8371
3,BD,1020,869
4,RR,723,713
5,DSQ,619,603
6,RO,463,449
7,SU,371,361
8,REF,291,283
9,CO,77,66


In [7]:
numeric_pos_values = pd.read_sql_query(
    f"""
    SELECT
        CAST(pos AS INTEGER) AS raw_pos,
        COUNT(*) AS rows,
        COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races,
        MIN(ran) AS min_ran,
        MAX(ran) AS max_ran
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'integer'
    GROUP BY CAST(pos AS INTEGER)
    ORDER BY raw_pos
    """,
    connection,
)

print(f"Distinct numeric pos values: {len(numeric_pos_values):,}")
numeric_pos_values

Distinct numeric pos values: 35


,raw_pos,rows,provisional_races,min_ran,max_ran
0,0,8,2,13,15
1,1,189344,189043,1,40
2,2,189007,188647,2,40
3,3,188141,187715,3,40
4,4,184662,184205,4,40
5,5,175886,175474,5,40
6,6,161327,161029,6,40
7,7,142993,142767,6,40
8,8,122268,122106,4,40
9,9,101726,101605,6,40


In [8]:
numeric_position_anomalies = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        ran,
        num,
        pos,
        btn,
        ovr_btn,
        horse,
        jockey,
        trainer,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'integer'
      AND (
          CAST(pos AS INTEGER) = 0
          OR CAST(pos AS INTEGER) > ran
      )
    ORDER BY
        date,
        course,
        off,
        CAST(pos AS INTEGER),
        rowid
    """,
    connection,
)

print(f"Numeric position anomalies: {len(numeric_position_anomalies):,}")
numeric_position_anomalies

Numeric position anomalies: 18


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,40679,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Rikiai Kurofune (JPN),Mitsuki Kaneko,Tsuyoshi Tanaka,
1,40680,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Fire (JPN),Kazuma Mori,Masaru Honda,
2,40681,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Shonan Coming (JPN),Taro Kusano,Tsuyoshi Tanaka,
3,40682,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Tenjin Kiyomori (JPN),Yusuke Eda,Ryo Takei,
4,40683,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Country Snow (JPN),Kazuma Harada,Hironori Kurita,
5,77097,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Champi...,Flat,13,1,0,0.00,0.00,Bank Of Burden (USA),Per-Anders Graberg,Niels Petersen,
6,77098,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Champi...,Flat,13,10,0,0.00,0.00,Mekong River (IRE),Oliver Wilson,Bettina Andersen,
7,77099,2015-06-28,Klampenborg (DEN),630668,3:00,Dansk Hesteforsikring Scandinavian Open Champi...,Flat,13,2,0,0.00,0.00,Moe Green (IRE),Rafael Schistl,Francisco Castro,
8,270486,2016-09-23,Haydock,657954,5:05,Griffiths And Armour Handicap,Flat,13,14,14,3.00,19.75,Jack Of Diamonds (IRE),Oisin Murphy,Roger Teal,In touch on outer - ridden along over 3f out -...
9,330864,2017-02-12,Gavea (BRZ),669090,7:45,Grande Premio Henrique Possollo (3yo Fillies)...,Flat,18,8,19,0.75,15.50,Pista Olimpica (BRZ),W Blandi,R Solanes,


In [9]:
affected_races = pd.read_sql_query(
    f"""
    WITH anomalous_races AS (
        SELECT DISTINCT
            date,
            course,
            off
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND (
              CAST(pos AS INTEGER) = 0
              OR CAST(pos AS INTEGER) > ran
          )
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.type,
        d.ran,
        d.num,
        d.pos,
        d.btn,
        d.ovr_btn,
        d.horse,
        d.jockey,
        d.trainer,
        d.comment
    FROM {SOURCE_TABLE} AS d
    INNER JOIN anomalous_races AS a
        ON d.date = a.date
       AND d.course = a.course
       AND d.off = a.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN 0
            ELSE 1
        END,
        CASE
            WHEN typeof(d.pos) = 'integer' THEN CAST(d.pos AS INTEGER)
            ELSE NULL
        END,
        d.rowid
    """,
    connection,
)

print(
    "Affected provisional races: "
    f"{affected_races[['date', 'course', 'off']].drop_duplicates().shape[0]:,}"
)
print(f"Runner records returned: {len(affected_races):,}")

affected_races

Affected provisional races: 11
Runner records returned: 119


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,40679,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Rikiai Kurofune (JPN),Mitsuki Kaneko,Tsuyoshi Tanaka,
1,40680,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Fire (JPN),Kazuma Mori,Masaru Honda,
2,40681,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Shonan Coming (JPN),Taro Kusano,Tsuyoshi Tanaka,
3,40682,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Tenjin Kiyomori (JPN),Yusuke Eda,Ryo Takei,
4,40683,2015-04-18,Nakayama (JPN),624701,7:40,Nakayama Grand Jump (Turf),Flat,15,,0,0.00,0.00,Country Snow (JPN),Kazuma Harada,Hironori Kurita,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,1813779,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),Flat,9,8,5,1.00,4.75,Nyaar (USA),Pat Dobbs,Doug Watson,Reared in stalls before start - led - prominen...
115,1813780,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),Flat,9,1,6,1.25,6.00,Shaq Diesel (USA),Richard Mullen,Bhupat Seemar,In touch with leaders - on outer when outpaced...
116,1813781,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),Flat,9,7,7,1.50,7.50,Laasudood (USA),Sandro Paiva,Bhupat Seemar,Prominent early - soon in touch with leaders -...
117,1813782,2026-03-13,Meydan,915628,17:45,Binghatti (Handicap) (Dirt),Flat,9,9,9,8.00,15.50,Quality Humor (USA),Connor Beasley,A bin Harmash,Reared in stalls before start - in rear - soon...


In [11]:
# Prepare row-level helper fields before summarising the affected races.
#
# This remains descriptive profiling only:
# - raw values are unchanged;
# - no anomaly classification is assigned;
# - pos > ran is not assumed to be erroneous.

affected_races_profile = affected_races.assign(
    # Preserve every observed position in a display-friendly text form,
    # including both numeric positions and textual outcome codes.
    pos_text=affected_races["pos"].astype(str),

    # Convert numeric position values to a nullable integer representation.
    # Textual outcome codes become missing values in this helper field.
    numeric_pos=pd.to_numeric(
        affected_races["pos"],
        errors="coerce",
    ).astype("Int64"),

    # Identify records whose raw runner-number field is blank.
    blank_num=affected_races["num"].astype(str).str.strip().eq(""),
)

# Compare each numeric position directly with the ran value on the same
# source row before performing the race-level aggregation.
affected_races_profile["pos_above_ran"] = (
    affected_races_profile["numeric_pos"]
    > affected_races_profile["ran"]
)

# Identify numeric zero-position records separately from textual outcomes.
affected_races_profile["zero_pos"] = (
    affected_races_profile["numeric_pos"].eq(0)
)

# Summarise each affected provisional race while retaining source descriptors
# needed for subsequent inspection.
affected_race_summary = (
    affected_races_profile
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "ran",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        # Count all source runner records returned for the race.
        source_rows=("source_rowid", "count"),

        # Count rows represented by a numeric position.
        numeric_pos_rows=("numeric_pos", "count"),

        # Count rows represented by a textual outcome code.
        text_pos_rows=("numeric_pos", lambda values: values.isna().sum()),

        # Count numeric zero-position records.
        zero_pos_rows=("zero_pos", "sum"),

        # Count numeric positions greater than the row-level ran value.
        pos_above_ran_rows=("pos_above_ran", "sum"),

        # Count records with a blank raw runner number.
        blank_num_rows=("blank_num", "sum"),

        # Show the complete observed position sequence for quick review.
        observed_positions=(
            "pos_text",
            lambda values: ", ".join(values.tolist()),
        ),
    )
)

affected_race_summary

,date,course,off,race_id,race_name,ran,source_rows,numeric_pos_rows,text_pos_rows,zero_pos_rows,pos_above_ran_rows,blank_num_rows,observed_positions
0,2015-04-18,Nakayama (JPN),7:40,624701,Nakayama Grand Jump (Turf),15,15,15,0,5,0,15,"0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10"
1,2015-06-28,Klampenborg (DEN),3:00,630668,Dansk Hesteforsikring Scandinavian Open Champi...,13,13,13,0,3,0,0,"0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10"
2,2016-09-23,Haydock,5:05,657954,Griffiths And Armour Handicap,13,13,13,0,0,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14"
3,2017-02-12,Gavea (BRZ),7:45,669090,Grande Premio Henrique Possollo (3yo Fillies)...,18,18,18,0,0,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
4,2017-02-23,Meydan (UAE),5:25,669525,Curlin Handicap Sponsored by Al Naboodah SUNWI...,9,9,9,0,0,1,0,"1, 2, 3, 4, 5, 6, 7, 9, 10"
5,2020-04-10,Gulfstream Park (USA),9:41,755688,Allowance Optional Claiming Race (Allowance) (...,8,8,8,0,0,1,0,"1, 2, 3, 5, 6, 7, 8, 9"
6,2020-08-15,Merano (ITY),5:55,765471,EBF Terme Di Merano () (3yo+) (Fillies & Mares...,6,6,6,0,0,2,6,"1, 2, 3, 4, 7, 9"
7,2023-12-09,Oaklawn Park (USA),10:14,857908,Mistletoe Stakes (Black Type) (Fillies & Mares...,4,4,4,0,0,1,4,"1, 2, 3, 8"
8,2023-12-23,Gulfstream Park (USA),9:36,857782,Mr. Prospector Stakes (3yo+) (Main Track) (Dirt),8,8,8,0,0,1,0,"1, 2, 3, 4, 6, 7, 8, 9"
9,2024-02-04,Tokyo (JPN),6:45,860553,Tokyo Shimbun Hai (4yo+) (Turf),16,16,16,0,0,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."


In [12]:
# Profile duplicate numeric finishing positions at the provisional race grain.
#
# A duplicate numeric position may represent a dead heat, an amended result,
# or a source anomaly. This cell only counts the observed pattern and does not
# yet assign any interpretation.

duplicate_numeric_positions = pd.read_sql_query(
    f"""
    WITH numeric_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runners_at_position
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
    )
    SELECT
        numeric_pos,
        COUNT(*) AS race_position_groups,
        SUM(runners_at_position) AS runner_records,
        MIN(runners_at_position) AS min_runners_at_position,
        MAX(runners_at_position) AS max_runners_at_position
    FROM numeric_positions
    WHERE runners_at_position > 1
    GROUP BY numeric_pos
    ORDER BY
        numeric_pos,
        race_position_groups DESC
    """,
    connection,
)

print(
    "Numeric finishing positions duplicated within a provisional race: "
    f"{duplicate_numeric_positions['race_position_groups'].sum():,}"
)

duplicate_numeric_positions

Numeric finishing positions duplicated within a provisional race: 3,007


,numeric_pos,race_position_groups,runner_records,min_runners_at_position,max_runners_at_position
0,1,301,602,2,2
1,2,360,720,2,2
2,3,426,852,2,2
3,4,455,912,2,3
4,5,410,822,2,3
5,6,296,594,2,3
6,7,225,451,2,3
7,8,161,323,2,3
8,9,121,242,2,2
9,10,85,170,2,2


In [13]:
# Test whether duplicated numeric positions follow the usual ordinal pattern
# expected after a dead heat.
#
# Example:
# - two runners recorded as position 1 would normally be followed by position 3;
# - three runners recorded as position 4 would normally be followed by position 7.
#
# This cell profiles the observed sequence only. It does not yet classify every
# duplicate as a confirmed dead heat.

duplicate_position_sequence_check = pd.read_sql_query(
    f"""
    WITH numeric_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runners_at_position
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
    ),
    duplicated_positions AS (
        SELECT
            date,
            course,
            off,
            numeric_pos,
            runners_at_position,
            numeric_pos + runners_at_position AS expected_next_position
        FROM numeric_positions
        WHERE runners_at_position > 1
    )
    SELECT
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM numeric_positions AS next_position
                WHERE next_position.date = duplicated_positions.date
                  AND next_position.course = duplicated_positions.course
                  AND next_position.off = duplicated_positions.off
                  AND next_position.numeric_pos =
                      duplicated_positions.expected_next_position
            )
            THEN 'expected_next_position_present'

            WHEN EXISTS (
                SELECT 1
                FROM numeric_positions AS immediate_position
                WHERE immediate_position.date = duplicated_positions.date
                  AND immediate_position.course = duplicated_positions.course
                  AND immediate_position.off = duplicated_positions.off
                  AND immediate_position.numeric_pos =
                      duplicated_positions.numeric_pos + 1
            )
            THEN 'immediate_next_position_present'

            ELSE 'no_later_numeric_position_observed'
        END AS sequence_pattern,

        COUNT(*) AS duplicated_race_position_groups,

        SUM(runners_at_position) AS runner_records

    FROM duplicated_positions
    GROUP BY sequence_pattern
    ORDER BY duplicated_race_position_groups DESC
    """,
    connection,
)

duplicate_position_sequence_check

,sequence_pattern,duplicated_race_position_groups,runner_records
0,expected_next_position_present,2929,5865
1,no_later_numeric_position_observed,78,157


In [14]:
# Test whether runners sharing the same positive numeric position also share
# the same cumulative beaten-distance value.
#
# Matching ovr_btn values would provide independent support that duplicated
# positions represent tied finishing outcomes rather than accidental duplicate
# rank assignments.
#
# Exact raw values remain unchanged, and no candidate dead-heat flag is created
# in this cell.

duplicate_position_distance_check = pd.read_sql_query(
    f"""
    WITH duplicated_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
        HAVING COUNT(*) > 1
    ),
    tied_runner_groups AS (
        SELECT
            d.date,
            d.course,
            d.off,
            CAST(d.pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runner_records,
            COUNT(DISTINCT CAST(d.ovr_btn AS TEXT)) AS distinct_ovr_btn_values,
            COUNT(DISTINCT CAST(d.btn AS TEXT)) AS distinct_btn_values
        FROM {SOURCE_TABLE} AS d
        INNER JOIN duplicated_positions AS p
            ON d.date = p.date
           AND d.course = p.course
           AND d.off = p.off
           AND CAST(d.pos AS INTEGER) = p.numeric_pos
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
        GROUP BY
            d.date,
            d.course,
            d.off,
            CAST(d.pos AS INTEGER)
    )
    SELECT
        CASE
            WHEN distinct_ovr_btn_values = 1
            THEN 'shared_ovr_btn'
            ELSE 'different_ovr_btn'
        END AS ovr_btn_pattern,
        COUNT(*) AS duplicated_race_position_groups,
        SUM(runner_records) AS runner_records,
        MIN(distinct_btn_values) AS min_distinct_btn_values,
        MAX(distinct_btn_values) AS max_distinct_btn_values
    FROM tied_runner_groups
    GROUP BY ovr_btn_pattern
    ORDER BY duplicated_race_position_groups DESC
    """,
    connection,
)

duplicate_position_distance_check

,ovr_btn_pattern,duplicated_race_position_groups,runner_records,min_distinct_btn_values,max_distinct_btn_values
0,shared_ovr_btn,3006,6020,1,2
1,different_ovr_btn,1,2,2,2


In [15]:
# Retrieve the only duplicated positive numeric position group whose runners
# do not share the same raw cumulative beaten-distance value.
#
# This isolates the exception for direct source-record inspection before any
# candidate dead-heat mapping is proposed.

duplicate_position_distance_exception = pd.read_sql_query(
    f"""
    WITH duplicated_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
        HAVING COUNT(*) > 1
    ),
    distance_exceptions AS (
        SELECT
            d.date,
            d.course,
            d.off,
            CAST(d.pos AS INTEGER) AS numeric_pos
        FROM {SOURCE_TABLE} AS d
        INNER JOIN duplicated_positions AS p
            ON d.date = p.date
           AND d.course = p.course
           AND d.off = p.off
           AND CAST(d.pos AS INTEGER) = p.numeric_pos
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
        GROUP BY
            d.date,
            d.course,
            d.off,
            CAST(d.pos AS INTEGER)
        HAVING COUNT(DISTINCT CAST(d.ovr_btn AS TEXT)) > 1
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.type,
        d.ran,
        d.num,
        d.pos,
        d.btn,
        d.ovr_btn,
        d.horse,
        d.jockey,
        d.trainer,
        d.comment
    FROM {SOURCE_TABLE} AS d
    INNER JOIN distance_exceptions AS e
        ON d.date = e.date
       AND d.course = e.course
       AND d.off = e.off
       AND CAST(d.pos AS INTEGER) = e.numeric_pos
    WHERE d.rowid <> 1
      AND typeof(d.pos) = 'integer'
    ORDER BY
        d.date,
        d.course,
        d.off,
        d.rowid
    """,
    connection,
)

duplicate_position_distance_exception

,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,55514,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,13,10,3.0,7.50,Baja Moon (AUS),Ben Melham,Patrick Payne,
1,55516,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,8,10,0.5,8.75,Cinnamon Carter (AUS),Joe Bowditch,Robbie Laing,


In [16]:
# Retrieve every runner from the Morphettville race containing the only
# duplicated positive position with inconsistent cumulative distances.
#
# The full race sequence is needed to inspect:
# - whether another ordinal position is missing;
# - how btn and ovr_btn progress around the duplicated position;
# - whether the duplicate may reflect a displaced or amended result.
#
# This remains source profiling only. No correction is applied.

morphettville_duplicate_exception_race = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        type,
        ran,
        num,
        pos,
        btn,
        ovr_btn,
        horse,
        jockey,
        trainer,
        comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND date = '2015-05-16'
      AND course = 'Morphettville (AUS)'
      AND off = '4:38'
    ORDER BY
        CASE
            WHEN typeof(pos) = 'integer'
            THEN CAST(pos AS INTEGER)
            ELSE 999
        END,
        rowid
    """,
    connection,
)

morphettville_duplicate_exception_race

,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,55238,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,7,1,0.00,0.00,Okahu Bay (AUS),Matthew Neilson,Phillip Stokes,
1,55239,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,1,2,0.10,0.10,Bahamas (AUS),Damien Oliver,Dan OSullivan,
2,55240,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,2,3,0.50,0.50,Ungrateful Ellen (AUS),Craig A Williams,Robert Smerdon,
3,55241,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,10,4,0.75,1.25,Keep The Klass (NZ),Michelle Payne,Darren Weir,
4,55243,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,4,5,0.30,1.75,Snow Secret (NZ),Glen Boss,Shaune Ritchie,
5,55301,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,3,6,0.75,2.50,Try Your Best (NZ),Dominic Tourneur,Phillip Stokes,
6,55379,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,6,7,1.00,3.50,Monopole (AUS),Jason Holder,Tony McEvoy,
7,55565,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,12,8,0.50,4.00,Kiwi Colleen (NZ),Mark Zahra,Andrew Noblet,
8,55381,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,14,9,0.50,4.50,Dane Hussler (AUS),Vlad Duric,David A Hayes & Tom Dabernig,
9,55514,2015-05-16,Morphettville (AUS),627591,4:38,William Hill SA Fillies Classic (3yo Fillies)...,Flat,14,13,10,3.00,7.50,Baja Moon (AUS),Ben Melham,Patrick Payne,


In [17]:
# Inventory every textual representation found in the two beaten-distance fields.
#
# Earlier storage profiling showed that btn and ovr_btn can be stored as text,
# particularly on rows where pos contains a non-finish outcome code.
#
# This cell records the exact raw textual values and their frequency without
# assigning meanings or converting them to candidate numeric values.

text_distance_values = pd.read_sql_query(
    f"""
    SELECT
        distance_field,
        raw_value,
        COUNT(*) AS rows,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races
    FROM (
        SELECT
            date,
            course,
            off,
            'btn' AS distance_field,
            TRIM(CAST(btn AS TEXT)) AS raw_value
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(btn) = 'text'

        UNION ALL

        SELECT
            date,
            course,
            off,
            'ovr_btn' AS distance_field,
            TRIM(CAST(ovr_btn AS TEXT)) AS raw_value
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(ovr_btn) = 'text'
    )
    GROUP BY
        distance_field,
        raw_value
    ORDER BY
        distance_field,
        rows DESC,
        raw_value
    """,
    connection,
)

print(
    "Distinct textual btn values: "
    f"{text_distance_values.loc[text_distance_values['distance_field'] == 'btn', 'raw_value'].nunique():,}"
)
print(
    "Distinct textual ovr_btn values: "
    f"{text_distance_values.loc[text_distance_values['distance_field'] == 'ovr_btn', 'raw_value'].nunique():,}"
)

text_distance_values

Distinct textual btn values: 1
Distinct textual ovr_btn values: 1


,distance_field,raw_value,rows,provisional_races
0,btn,-,93992,43388
1,ovr_btn,-,93992,43388


In [18]:
# Cross-tabulate each textual position code against the observed beaten-distance
# representations on the same source rows.
#
# This will show whether all non-finish codes use "-" consistently, or whether
# particular outcomes such as disqualification retain numeric distance values.
#
# The cell remains descriptive: no outcome meanings or candidate mappings are
# assigned yet.

text_pos_distance_patterns = pd.read_sql_query(
    f"""
    SELECT
        TRIM(CAST(pos AS TEXT)) AS raw_pos,
        typeof(btn) AS btn_storage,
        TRIM(CAST(btn AS TEXT)) AS raw_btn,
        typeof(ovr_btn) AS ovr_btn_storage,
        TRIM(CAST(ovr_btn AS TEXT)) AS raw_ovr_btn,
        COUNT(*) AS rows,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
    GROUP BY
        TRIM(CAST(pos AS TEXT)),
        typeof(btn),
        TRIM(CAST(btn AS TEXT)),
        typeof(ovr_btn),
        TRIM(CAST(ovr_btn AS TEXT))
    ORDER BY
        raw_pos,
        rows DESC
    """,
    connection,
)

text_pos_distance_patterns

,raw_pos,btn_storage,raw_btn,ovr_btn_storage,raw_ovr_btn,rows,provisional_races
0,BD,text,-,text,-,1020,869
1,CO,text,-,text,-,77,66
2,DSQ,integer,0,integer,0,185,185
3,DSQ,real,0.2,real,0.2,7,7
4,DSQ,integer,1,integer,1,5,5
...,...,...,...,...,...,...,...
367,REF,text,-,text,-,291,283
368,RO,text,-,text,-,463,449
369,RR,text,-,text,-,723,713
370,SU,text,-,text,-,371,361


In [19]:
# Summarise how beaten-distance fields are represented for each textual
# finishing-position code.
#
# In particular, this distinguishes:
# - rows where both distance fields use the textual sentinel "-";
# - rows where both fields retain numeric values;
# - any mixed or inconsistent storage combinations.
#
# Numeric ranges are included for profiling only. No meaning or candidate
# mapping is assigned to the position codes in this cell.

text_pos_distance_summary = pd.read_sql_query(
    f"""
    SELECT
        TRIM(CAST(pos AS TEXT)) AS raw_pos,

        COUNT(*) AS rows,

        SUM(
            CASE
                WHEN typeof(btn) = 'text'
                 AND TRIM(CAST(btn AS TEXT)) = '-'
                 AND typeof(ovr_btn) = 'text'
                 AND TRIM(CAST(ovr_btn AS TEXT)) = '-'
                THEN 1
                ELSE 0
            END
        ) AS dash_distance_rows,

        SUM(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                 AND typeof(ovr_btn) IN ('integer', 'real')
                THEN 1
                ELSE 0
            END
        ) AS numeric_distance_rows,

        SUM(
            CASE
                WHEN NOT (
                    (
                        typeof(btn) = 'text'
                        AND TRIM(CAST(btn AS TEXT)) = '-'
                        AND typeof(ovr_btn) = 'text'
                        AND TRIM(CAST(ovr_btn AS TEXT)) = '-'
                    )
                    OR
                    (
                        typeof(btn) IN ('integer', 'real')
                        AND typeof(ovr_btn) IN ('integer', 'real')
                    )
                )
                THEN 1
                ELSE 0
            END
        ) AS mixed_or_other_rows,

        MIN(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                THEN CAST(btn AS REAL)
            END
        ) AS min_numeric_btn,

        MAX(
            CASE
                WHEN typeof(btn) IN ('integer', 'real')
                THEN CAST(btn AS REAL)
            END
        ) AS max_numeric_btn,

        MIN(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN CAST(ovr_btn AS REAL)
            END
        ) AS min_numeric_ovr_btn,

        MAX(
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN CAST(ovr_btn AS REAL)
            END
        ) AS max_numeric_ovr_btn

    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
    GROUP BY TRIM(CAST(pos AS TEXT))
    ORDER BY rows DESC, raw_pos
    """,
    connection,
)

text_pos_distance_summary

,raw_pos,rows,dash_distance_rows,numeric_distance_rows,mixed_or_other_rows,min_numeric_btn,max_numeric_btn,min_numeric_ovr_btn,max_numeric_ovr_btn
0,PU,65832,65832,0,0,NaN,NaN,NaN,NaN
1,F,15681,15681,0,0,NaN,NaN,NaN,NaN
2,UR,9527,9527,0,0,NaN,NaN,NaN,NaN
3,BD,1020,1020,0,0,NaN,NaN,NaN,NaN
4,RR,723,723,0,0,NaN,NaN,NaN,NaN
5,DSQ,619,0,619,0,0.0,79.0,0.0,198.0
6,RO,463,463,0,0,NaN,NaN,NaN,NaN
7,SU,371,371,0,0,NaN,NaN,NaN,NaN
8,REF,291,291,0,0,NaN,NaN,NaN,NaN
9,CO,77,77,0,0,NaN,NaN,NaN,NaN


In [20]:
# Profile disqualified runners in their full race context.
#
# The aim is to determine whether DSQ records commonly retain:
# - a measured finishing distance;
# - an implied finishing order from ovr_btn;
# - explanatory wording about original or amended placings.
#
# No candidate finishing position is inferred here. The raw DSQ outcome remains
# authoritative and all numeric distance values are preserved unchanged.

dsq_profile = pd.read_sql_query(
    f"""
    WITH race_rows AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            ran,
            num,
            pos,
            btn,
            ovr_btn,
            horse,
            comment,

            -- Rank every runner with a numeric cumulative distance within the
            -- provisional race. Equal distances receive the same dense rank.
            CASE
                WHEN typeof(ovr_btn) IN ('integer', 'real')
                THEN DENSE_RANK() OVER (
                    PARTITION BY date, course, off
                    ORDER BY CAST(ovr_btn AS REAL)
                )
            END AS distance_rank

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
    )
    SELECT
        source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        ran,
        num,
        pos,
        btn,
        ovr_btn,
        distance_rank,
        horse,
        comment,

        CASE
            WHEN LOWER(comment) LIKE '%finished%'
              OR LOWER(comment) LIKE '%placed%'
              OR LOWER(comment) LIKE '%disqual%'
              OR LOWER(comment) LIKE '%demoted%'
              OR LOWER(comment) LIKE '%promoted%'
            THEN 1
            ELSE 0
        END AS explicit_result_wording

    FROM race_rows
    WHERE typeof(pos) = 'text'
      AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
    ORDER BY
        explicit_result_wording DESC,
        date,
        course,
        off,
        source_rowid
    """,
    connection,
)

print(f"DSQ runner records: {len(dsq_profile):,}")
print(
    "DSQ rows with explicit result wording in comment: "
    f"{dsq_profile['explicit_result_wording'].sum():,}"
)

dsq_profile

DSQ runner records: 619
DSQ rows with explicit result wording in comment: 598


,source_rowid,date,course,off,race_id,race_name,ran,num,pos,btn,ovr_btn,distance_rank,horse,comment,explicit_result_wording
0,4550,2015-01-15,Pornichet-La Baule (FR),8:25,617550,Prix dAbaca (Conditions) (4yo) (Viscoride),10,1,DSQ,7.00,18.25,7,Inseo (FR),Finished 7th - disqualified and placed last - ...,1
1,14783,2015-02-15,Navan (IRE),3:55,619171,Ten Up Novice Chase,6,6,DSQ,0.00,0.00,1,Very Wood (FR),Tracked leaders until disputed from 7th and le...,1
2,15823,2015-02-19,Meydan (UAE),5:05,619483,District One (Handicap) (Dirt),15,9,DSQ,0.00,0.00,1,Storm Belt (USA),Mid-division - smooth progress to chase leader...,1
3,16169,2015-02-20,Jebel Ali (UAE),11:35,619514,Jebel Ali Mile Sponsored By Derrinstown Stud ...,9,5,DSQ,0.00,0.00,1,Forjatt (IRE),In rear of mid-division - smooth progress 3f o...,1
4,19221,2015-02-28,Meydan (UAE),3:20,619987,IPIC 30th Anniversary (Handicap) (Dirt),8,1,DSQ,0.00,0.00,1,Jeeraan (USA),Tracked leader - led 4f out - ridden 2 1/2 out...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614,1658807,2025-04-11,Saint-Cloud (FR),4:30,893213,Prix du Lieu Marmion (Handicap) (4yo) (Turf),12,3,DSQ,1.75,11.25,8,Iberia (FR),,0
615,1742377,2025-10-01,Auteuil (FR),11:17,904979,Prix Vivienne (Hurdle) (Claimer) (5yo+) (Turf),7,7,DSQ,30.00,68.75,7,Xat (FR),,0
616,1790299,2026-01-11,Sha Tin,09:50,911170,Stanley Gap Handicap (3yo+) (Course C+3) (Turf),14,6,DSQ,0.05,0.05,2,Stormy Grove (AUS),Midfield - headway over 1f out - joined winner...,0
617,1826039,2026-04-09,Auteuil,14:05,917329,Prix Aladdin (Conditions Hurdle) (Turf),9,9,DSQ,1.25,1.25,2,Varsovie (FR),,0


In [21]:
# Summarise recurring result-amendment wording in comments attached to DSQ rows.
#
# Categories are deliberately descriptive rather than mutually interpretive.
# A single comment may match more than one wording flag, so the counts should
# not be expected to sum to the total number of DSQ records.
#
# This cell does not infer an original or final numeric placing.

dsq_comment_wording_summary = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS dsq_rows,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%finished%'
                THEN 1 ELSE 0
            END
        ) AS mentions_finished,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%placed last%'
                THEN 1 ELSE 0
            END
        ) AS mentions_placed_last,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%disqual%'
                THEN 1 ELSE 0
            END
        ) AS mentions_disqualified,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%demoted%'
                THEN 1 ELSE 0
            END
        ) AS mentions_demoted,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%promoted%'
                THEN 1 ELSE 0
            END
        ) AS mentions_promoted,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%weighed in%'
                  OR LOWER(comment) LIKE '%weighed-in%'
                  OR LOWER(comment) LIKE '%weigh in%'
                THEN 1 ELSE 0
            END
        ) AS mentions_weighing,

        SUM(
            CASE
                WHEN TRIM(comment) = ''
                THEN 1 ELSE 0
            END
        ) AS blank_comments,

        SUM(
            CASE
                WHEN TRIM(comment) <> ''
                 AND LOWER(comment) NOT LIKE '%finished%'
                 AND LOWER(comment) NOT LIKE '%placed%'
                 AND LOWER(comment) NOT LIKE '%disqual%'
                 AND LOWER(comment) NOT LIKE '%demoted%'
                 AND LOWER(comment) NOT LIKE '%promoted%'
                THEN 1 ELSE 0
            END
        ) AS nonblank_without_explicit_result_wording

    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
      AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
    """,
    connection,
)

dsq_comment_wording_summary


,dsq_rows,mentions_finished,mentions_placed_last,mentions_disqualified,mentions_demoted,mentions_promoted,mentions_weighing,blank_comments,nonblank_without_explicit_result_wording
0,619,490,117,594,0,0,126,14,7


In [22]:
# Profile the non-DSQ textual position codes across the source.
#
# These codes currently share the same distance representation:
# btn = "-" and ovr_btn = "-".
#
# This cell examines their surrounding source context without assigning final
# meanings or candidate mappings. In particular, it records:
# - the range of ran values;
# - the balance of recorded race types;
# - whether runner numbers are blank;
# - how many provisional races contain each code.

non_dsq_text_pos_profile = pd.read_sql_query(
    f"""
    SELECT
        TRIM(CAST(pos AS TEXT)) AS raw_pos,

        COUNT(*) AS runner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        MIN(ran) AS min_ran,
        MAX(ran) AS max_ran,

        SUM(
            CASE
                WHEN TRIM(CAST(num AS TEXT)) = ''
                THEN 1
                ELSE 0
            END
        ) AS blank_num_rows,

        SUM(
            CASE
                WHEN type = 'Flat'
                THEN 1
                ELSE 0
            END
        ) AS flat_rows,

        SUM(
            CASE
                WHEN type = 'Hurdle'
                THEN 1
                ELSE 0
            END
        ) AS hurdle_rows,

        SUM(
            CASE
                WHEN type = 'Chase'
                THEN 1
                ELSE 0
            END
        ) AS chase_rows,

        SUM(
            CASE
                WHEN type NOT IN ('Flat', 'Hurdle', 'Chase')
                THEN 1
                ELSE 0
            END
        ) AS other_type_rows

    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
      AND TRIM(CAST(pos AS TEXT)) <> 'DSQ'
    GROUP BY TRIM(CAST(pos AS TEXT))
    ORDER BY runner_records DESC, raw_pos
    """,
    connection,
)

non_dsq_text_pos_profile

,raw_pos,runner_records,provisional_races,min_ran,max_ran,blank_num_rows,flat_rows,hurdle_rows,chase_rows,other_type_rows
0,PU,65832,33433,2,40,38,2159,36580,26422,671
1,F,15681,12920,2,40,13,314,7429,7899,39
2,UR,9527,8371,2,40,9,882,3654,4889,102
3,BD,1020,869,3,40,1,46,577,374,23
4,RR,723,713,3,27,11,429,194,86,14
5,RO,463,449,3,27,0,28,255,125,55
6,SU,371,361,3,29,1,75,203,45,48
7,REF,291,283,3,40,0,18,93,180,0
8,CO,77,66,5,25,0,0,42,32,3
9,LFT,7,7,7,17,0,5,1,1,0


In [23]:
# Compare the recorded race-level ran value with the number and type of
# source runner records present in each provisional race.
#
# This tests whether ran represents:
# - all source runner records;
# - only runners with numeric finishing positions;
# - or another source convention.
#
# The cell reports patterns only. It does not redefine ran or correct any race.

race_result_completeness = pd.read_sql_query(
    f"""
    WITH race_result_counts AS (
        SELECT
            date,
            course,
            off,

            MIN(ran) AS min_ran,
            MAX(ran) AS max_ran,

            COUNT(*) AS source_rows,

            SUM(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) > 0
                    THEN 1
                    ELSE 0
                END
            ) AS positive_numeric_pos_rows,

            SUM(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) = 0
                    THEN 1
                    ELSE 0
                END
            ) AS zero_pos_rows,

            SUM(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
                    THEN 1
                    ELSE 0
                END
            ) AS dsq_rows,

            SUM(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) <> 'DSQ'
                    THEN 1
                    ELSE 0
                END
            ) AS other_text_outcome_rows

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        CASE
            WHEN min_ran <> max_ran
            THEN 'ran_inconsistent_within_race'

            WHEN source_rows = min_ran
            THEN 'source_rows_equal_ran'

            WHEN source_rows < min_ran
            THEN 'source_rows_below_ran'

            ELSE 'source_rows_above_ran'
        END AS race_pattern,

        COUNT(*) AS provisional_races,

        SUM(source_rows) AS source_runner_records,

        SUM(positive_numeric_pos_rows) AS positive_numeric_pos_rows,

        SUM(zero_pos_rows) AS zero_pos_rows,

        SUM(dsq_rows) AS dsq_rows,

        SUM(other_text_outcome_rows) AS other_text_outcome_rows,

        MIN(source_rows - min_ran) AS min_row_difference,

        MAX(source_rows - max_ran) AS max_row_difference

    FROM race_result_counts
    GROUP BY race_pattern
    ORDER BY provisional_races DESC
    """,
    connection,
)

race_result_completeness

,race_pattern,provisional_races,source_runner_records,positive_numeric_pos_rows,zero_pos_rows,dsq_rows,other_text_outcome_rows,min_row_difference,max_row_difference
0,source_rows_equal_ran,189038,1851253,1756634,8,619,93992,0,0
1,source_rows_below_ran,5,32,32,0,0,0,-4,-1


In [24]:
# Retrieve the five provisional races where the number of source runner records
# is lower than the recorded ran value.
#
# These races contain only positive numeric positions, so their row deficits
# are not explained by textual non-finish outcomes, DSQ records, or pos = 0.
#
# The full result sequences are returned for direct inspection before deciding
# whether the source extract is incomplete or follows another convention.

races_below_recorded_ran = pd.read_sql_query(
    f"""
    WITH race_counts AS (
        SELECT
            date,
            course,
            off,
            MIN(ran) AS recorded_ran,
            MAX(ran) AS max_recorded_ran,
            COUNT(*) AS source_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
        HAVING COUNT(*) < MIN(ran)
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.type,
        d.ran,
        c.source_rows,
        d.ran - c.source_rows AS missing_row_count,
        d.num,
        d.pos,
        d.btn,
        d.ovr_btn,
        d.horse,
        d.jockey,
        d.trainer,
        d.comment
    FROM {SOURCE_TABLE} AS d
    INNER JOIN race_counts AS c
        ON d.date = c.date
       AND d.course = c.course
       AND d.off = c.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.pos) = 'integer'
            THEN CAST(d.pos AS INTEGER)
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

print(
    "Affected provisional races: "
    f"{races_below_recorded_ran[['date', 'course', 'off']].drop_duplicates().shape[0]:,}"
)
print(f"Runner records returned: {len(races_below_recorded_ran):,}")

races_below_recorded_ran

Affected provisional races: 5
Runner records returned: 32


,source_rowid,date,course,race_id,off,race_name,type,ran,source_rows,missing_row_count,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,1524983,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,1,0.00,0.00,Pyrrhaa (FR),Paulin Blot,Mme Brigitte Re-Scandella,
1,1524982,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,2,2.50,2.50,Lhubert De Houelle (FR),Mme Cathy Joubert,A Chaille-Chaille,
2,1524981,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,3,2.50,5.00,Bakarelo (IRE),Leo-Paul Brechet,Gabriel Leenders,
3,1524980,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,4,6.00,11.00,Shalez (FR),Gaetan Champier,Mme Brigitte Re-Scandella,
4,1524979,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,5,2.00,13.00,Bonham Strand (FR),Francesco Mula,Francois Nicolle,
5,1524978,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,6,10.00,23.00,Lulu Stars (FR),Gwen Richard,Mme V Seignoux,
6,1524977,2024-06-18,Nantes (FR),873374,2:14,Prix Sarah Gosse (Conditions Hurdle) (Turf),Hurdle,8,7,1,,7,30.00,53.00,Cockling (FR),Stephane Paillard,J Jouin,
7,1528630,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),Flat,5,4,1,12,1,0.00,0.00,Kings Sword (JPN),Yusuke Fujioka,Ryo Terashima,
8,1528614,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),Flat,5,4,1,8,2,1.75,1.75,Wilson Tesoro (JPN),Yuga Kawada,Hitoshi Kotegawa,
9,1528615,2024-06-26,Ohi (JPN),871722,11:07,Teio Sho (Local (Dirt),Flat,5,4,1,3,3,1.00,2.75,Diktaean (JPN),Kazuo Yokoyama,Tatsuya Yoshioka,


In [25]:
# Produce one compact race-level summary for the five races where the number
# of source rows is lower than the recorded ran value.
#
# This confirms whether each extract contains a continuous sequence of positive
# positions and shows exactly where the observed result stops.
#
# No missing runners or outcomes are inferred.

races_below_ran_summary = (
    races_below_recorded_ran
    .assign(
        # Convert the raw position to numeric for sequence checks.
        numeric_pos=pd.to_numeric(
            races_below_recorded_ran["pos"],
            errors="coerce",
        ).astype("Int64"),

        # Identify blank raw runner-number values.
        blank_num=races_below_recorded_ran["num"]
        .astype(str)
        .str.strip()
        .eq(""),
    )
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "ran",
            "source_rows",
            "missing_row_count",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        # Record the lowest and highest observed numeric positions.
        min_observed_pos=("numeric_pos", "min"),
        max_observed_pos=("numeric_pos", "max"),

        # Count distinct observed numeric positions.
        distinct_observed_positions=("numeric_pos", "nunique"),

        # Count blank raw runner numbers.
        blank_num_rows=("blank_num", "sum"),

        # Preserve the complete observed position sequence for inspection.
        observed_positions=(
            "numeric_pos",
            lambda values: ", ".join(
                str(int(value))
                for value in values.dropna().sort_values()
            ),
        ),
    )
)

# Test whether each race contains a continuous sequence from position 1
# through the highest observed position.
races_below_ran_summary["continuous_from_first"] = (
    races_below_ran_summary["min_observed_pos"].eq(1)
    & races_below_ran_summary["distinct_observed_positions"].eq(
        races_below_ran_summary["max_observed_pos"]
    )
)

races_below_ran_summary

,date,course,off,race_id,race_name,ran,source_rows,missing_row_count,min_observed_pos,max_observed_pos,distinct_observed_positions,blank_num_rows,observed_positions,continuous_from_first
0,2024-06-18,Nantes (FR),2:14,873374,Prix Sarah Gosse (Conditions Hurdle) (Turf),8,7,1,1,7,7,7,"1, 2, 3, 4, 5, 6, 7",True
1,2024-06-26,Ohi (JPN),11:07,871722,Teio Sho (Local (Dirt),5,4,1,1,4,4,0,"1, 2, 3, 4",True
2,2024-09-03,Morioka (JPN),11:07,876244,Kozukata Sho (Local (Dirt),5,4,1,1,4,4,0,"1, 2, 3, 4",True
3,2024-09-26,Funabashi (JPN),11:07,878244,Marine Cup (Local (Fillies) (Dirt),6,2,4,1,2,2,2,"1, 2",True
4,2025-10-09,Ohi (JPN),11:07,905803,Tokyo Hai (Local (Dirt),16,15,1,1,15,15,14,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15",True


In [26]:
# Test whether cumulative beaten distance (ovr_btn) progresses consistently
# from the preceding numeric finisher's cumulative distance plus btn.
#
# The first numeric finisher in a race is expected to have ovr_btn = 0.
# For later finishers, the expected relationship is:
#
#     current ovr_btn = previous ovr_btn + current btn
#
# Dead heats and anomalous position sequences are retained in the calculation.
# This cell only profiles exact agreement and disagreement; it does not correct
# any values or define final candidate distance attributes.

numeric_distance_consistency = pd.read_sql_query(
    f"""
    WITH numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,

            LAG(CAST(ovr_btn AS REAL)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_ovr_btn,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(btn) IN ('integer', 'real')
          AND typeof(ovr_btn) IN ('integer', 'real')
    )
    SELECT
        CASE
            WHEN numeric_finish_order = 1
             AND ABS(numeric_ovr_btn) < 0.000001
            THEN 'first_finisher_zero'

            WHEN numeric_finish_order = 1
            THEN 'first_finisher_nonzero'

            WHEN ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + numeric_btn)
            ) < 0.000001
            THEN 'later_finisher_exact_match'

            ELSE 'later_finisher_mismatch'
        END AS distance_pattern,

        COUNT(*) AS runner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races

    FROM numeric_finishers
    GROUP BY distance_pattern
    ORDER BY runner_records DESC
    """,
    connection,
)

numeric_distance_consistency

,distance_pattern,runner_records,provisional_races
0,later_finisher_exact_match,1265556,188936
1,later_finisher_mismatch,302067,125992
2,first_finisher_zero,188544,188544
3,first_finisher_nonzero,499,499


In [27]:
# Record the externally verified Morphettville source anomaly separately from
# the general profiling logic.
#
# This does not alter or overwrite the raw source record. It creates a
# notebook-level audit record describing:
# - the exact source row affected;
# - the observed raw values;
# - the externally supported result;
# - the handling decision for later staging work.
#
# Final schema design remains deferred.

externally_verified_source_anomalies = pd.DataFrame(
    [
        {
            "source_rowid": 55516,
            "date": "2015-05-16",
            "course": "Morphettville (AUS)",
            "off": "4:38",
            "race_id": 627591,
            "horse": "Cinnamon Carter (AUS)",

            # Exact raw source values.
            "raw_pos": 10,
            "raw_btn": 0.50,
            "raw_ovr_btn": 8.75,

            # Externally supported published result.
            "verified_finish_position": 12,
            "verified_dead_heat": True,
            "verified_tied_with": "Mighty Maher (AUS)",
            "verified_position_13_skipped": True,

            # The published margin differs slightly across representation and
            # rounding conventions, so it is recorded descriptively rather
            # than used to overwrite the raw distance value.
            "verified_margin_note": (
                "Published result places Cinnamon Carter in a dead heat for "
                "12th with Mighty Maher; source position 10 is inconsistent."
            ),

            # External evidence retained for reproducibility and audit.
            "verification_source": "Breednet SA Fillies Classic 2015 result",
            "verification_url": (
                "https://www.breednet.com.au/stakes-race-results/"
                "race-history?racename=sajc+fillies+classic"
            ),
            "verification_status": "externally_verified",

            # Preserve first; flag later. No raw correction is made here.
            "handling_decision": (
                "Preserve raw values and attach a verified source-anomaly "
                "flag in a later staging layer."
            ),
        }
    ]
)

externally_verified_source_anomalies

,source_rowid,date,course,off,race_id,horse,raw_pos,raw_btn,raw_ovr_btn,verified_finish_position,verified_dead_heat,verified_tied_with,verified_position_13_skipped,verified_margin_note,verification_source,verification_url,verification_status,handling_decision
0,55516,2015-05-16,Morphettville (AUS),4:38,627591,Cinnamon Carter (AUS),10,0.5,8.75,12,True,Mighty Maher (AUS),True,Published result places Cinnamon Carter in a d...,Breednet SA Fillies Classic 2015 result,https://www.breednet.com.au/stakes-race-result...,externally_verified,Preserve raw values and attach a verified sour...


In [28]:
# Measure the size of the difference between recorded ovr_btn and the value
# implied by:
#
#     previous ovr_btn + current btn
#
# This profiles later positive numeric finishers only.
#
# The calculation does not assume that every non-zero difference is an error.
# Small differences may reflect:
# - fractional-margin rounding;
# - conversion between textual racing margins and numeric values;
# - jurisdiction-specific distance conventions;
# - ordering effects around dead heats;
# - amended or anomalous source results.

numeric_distance_difference_profile = pd.read_sql_query(
    f"""
    WITH numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,

            LAG(CAST(ovr_btn AS REAL)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_ovr_btn,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(btn) IN ('integer', 'real')
          AND typeof(ovr_btn) IN ('integer', 'real')
    ),
    later_finishers AS (
        SELECT
            *,
            ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + numeric_btn)
            ) AS absolute_difference
        FROM numeric_finishers
        WHERE numeric_finish_order > 1
    )
    SELECT
        CASE
            WHEN absolute_difference < 0.000001
            THEN 'exact'

            WHEN absolute_difference <= 0.05
            THEN 'above_0_to_0.05'

            WHEN absolute_difference <= 0.10
            THEN 'above_0.05_to_0.10'

            WHEN absolute_difference <= 0.25
            THEN 'above_0.10_to_0.25'

            WHEN absolute_difference <= 0.50
            THEN 'above_0.25_to_0.50'

            WHEN absolute_difference <= 1.00
            THEN 'above_0.50_to_1.00'

            ELSE 'above_1.00'
        END AS difference_band,

        COUNT(*) AS runner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        MIN(absolute_difference) AS min_absolute_difference,

        MAX(absolute_difference) AS max_absolute_difference

    FROM later_finishers
    GROUP BY difference_band
    ORDER BY
        CASE difference_band
            WHEN 'exact' THEN 1
            WHEN 'above_0_to_0.05' THEN 2
            WHEN 'above_0.05_to_0.10' THEN 3
            WHEN 'above_0.10_to_0.25' THEN 4
            WHEN 'above_0.25_to_0.50' THEN 5
            WHEN 'above_0.50_to_1.00' THEN 6
            ELSE 7
        END
    """,
    connection,
)

numeric_distance_difference_profile

,difference_band,runner_records,provisional_races,min_absolute_difference,max_absolute_difference
0,exact,1265556,188936,0.00,5.551115e-17
1,above_0_to_0.05,122020,74861,0.05,5.000000e-02
2,above_0.05_to_0.10,111759,78419,0.05,1.000000e-01
3,above_0.10_to_0.25,65183,44156,0.10,2.500000e-01
4,above_0.25_to_0.50,875,568,0.25,5.000000e-01
5,above_0.50_to_1.00,638,395,0.55,1.000000e+00
6,above_1.00,1592,990,1.05,1.520000e+02


In [29]:
# Inventory the most common non-zero arithmetic differences between:
#
#     recorded ovr_btn
#
# and:
#
#     previous ovr_btn + current btn
#
# This will show whether the mismatch population is concentrated at recurring
# fractional values such as 0.05, 0.10, and 0.25.
#
# Differences are rounded only for grouping and display. The underlying raw
# btn and ovr_btn values remain unchanged.

numeric_distance_difference_values = pd.read_sql_query(
    f"""
    WITH numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,

            LAG(CAST(ovr_btn AS REAL)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_ovr_btn,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(btn) IN ('integer', 'real')
          AND typeof(ovr_btn) IN ('integer', 'real')
    ),
    later_finishers AS (
        SELECT
            date,
            course,
            off,
            ROUND(
                ABS(
                    numeric_ovr_btn
                    - (previous_ovr_btn + numeric_btn)
                ),
                4
            ) AS rounded_absolute_difference
        FROM numeric_finishers
        WHERE numeric_finish_order > 1
    )
    SELECT
        rounded_absolute_difference,
        COUNT(*) AS runner_records,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races
    FROM later_finishers
    WHERE rounded_absolute_difference > 0
    GROUP BY rounded_absolute_difference
    ORDER BY
        runner_records DESC,
        rounded_absolute_difference
    LIMIT 30
    """,
    connection,
)

numeric_distance_difference_values

,rounded_absolute_difference,runner_records,provisional_races
0,0.05,211488,110702
1,0.10,39330,34425
2,0.20,31003,24622
3,0.15,16626,15631
4,0.50,531,321
5,0.25,518,510
6,0.75,303,182
7,0.30,287,256
8,1.00,255,171
9,1.25,204,125


In [30]:
# Retrieve the largest arithmetic mismatches between the recorded cumulative
# beaten distance and the value implied by:
#
#     previous ovr_btn + current btn
#
# Only later positive numeric finishers with an absolute difference above
# 1.00 are included.
#
# The result includes the preceding runner's position and cumulative distance
# so that ordering effects, dead heats, missing ranks, and obvious source
# contradictions can be inspected directly.
#
# No correction or candidate distance rule is applied.

large_numeric_distance_mismatches = pd.read_sql_query(
    f"""
    WITH numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            race_id,
            off,
            race_name,
            ran,
            num,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,
            horse,
            comment,

            LAG(CAST(pos AS INTEGER)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_numeric_pos,

            LAG(CAST(ovr_btn AS REAL)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_ovr_btn,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(btn) IN ('integer', 'real')
          AND typeof(ovr_btn) IN ('integer', 'real')
    )
    SELECT
        source_rowid,
        date,
        course,
        race_id,
        off,
        race_name,
        ran,
        num,
        numeric_pos,
        previous_numeric_pos,
        numeric_btn,
        previous_ovr_btn,
        numeric_ovr_btn,

        ROUND(
            previous_ovr_btn + numeric_btn,
            4
        ) AS implied_ovr_btn,

        ROUND(
            ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + numeric_btn)
            ),
            4
        ) AS absolute_difference,

        horse,
        comment

    FROM numeric_finishers
    WHERE numeric_finish_order > 1
      AND ABS(
          numeric_ovr_btn
          - (previous_ovr_btn + numeric_btn)
      ) > 1.00

    ORDER BY
        absolute_difference DESC,
        date,
        course,
        off,
        numeric_pos,
        source_rowid

    LIMIT 100
    """,
    connection,
)

print(
    "Large mismatches returned: "
    f"{len(large_numeric_distance_mismatches):,}"
)

large_numeric_distance_mismatches

Large mismatches returned: 100


,source_rowid,date,course,race_id,off,race_name,ran,num,numeric_pos,previous_numeric_pos,numeric_btn,previous_ovr_btn,numeric_ovr_btn,implied_ovr_btn,absolute_difference,horse,comment
0,553992,2018-06-12,Thirsk,702720,3:15,100% Racing UK Profits Back To Racing Handicap,16,2,16,15,182.00,13.75,43.75,195.75,152.0,Lexington Times (IRE),Effectively refused to race (starter reported ...
1,1701535,2025-07-04,Wexford (IRE),899278,2:40,Clearwater Construction Handicap Hurdle (Div II),14,5,14,13,4.00,84.00,4.00,88.00,84.0,Onebrightbluerose (IRE),Prominent - went second and pressed winner on ...
2,1017871,2021-05-28,Stratford,783577,5:10,PPSA Open Hunters Chase,9,4,7,6,0.00,72.50,0.00,72.50,72.5,Capitaine (FR),Prominent - led 3rd - went clear after 2 out -...
3,202772,2016-05-06,Lingfield (AW),648250,1:30,Heart FM Maiden Stakes,7,4,7,6,166.00,9.25,108.25,175.25,67.0,Northman (IRE),Pushed along after 3f - under strong pressure ...
4,203540,2016-05-07,Lingfield,648279,4:35,betfred.com Handicap,7,1,7,6,166.00,10.50,109.50,176.50,67.0,Slowfoot (GER),Steadied start - held up in last pair - lost t...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,710932,2019-05-10,Market Rasen,727447,3:20,Victor Lucas Memorial Handicap Hurdle,8,3,7,6,123.00,49.50,148.50,172.50,24.0,Satoshi (IRE),Held up - not fluent - weakened 8th(op 16/1)
96,1691983,2025-06-13,Goodwood,895155,5:30,Sir Eric Parker Memorial EBF Restricted Maiden...,9,4,9,8,123.00,13.50,112.50,136.50,24.0,Storm Terri (GB),Soon outpaced and detached - tailed off (jocke...
97,144036,2015-11-21,Delta downs (USA),639764,9:43,Delta Downs Princess Stakes (2yo Fillies) (Dirt),7,5,6,5,2.75,35.00,14.25,37.75,23.5,Shesthewinner (USA),Finished 4th - disqualified and placed 6th
98,524698,2018-04-23,Pontefract,698000,2:40,Napoleons Casino Bradford Handicap,15,3,14,13,122.00,32.75,131.75,154.75,23.0,Lomu (IRE),Led - ridden along and headed well over 2f out...


In [31]:
# Inventory unusually large numeric btn values.
#
# The largest arithmetic mismatches suggest that some numeric values may encode
# special racing-margin descriptions or jurisdiction-specific conventions
# rather than literal incremental lengths.
#
# This cell profiles values greater than 50 and provides:
# - their frequency;
# - the number of races and courses involved;
# - their observed ovr_btn range;
# - counts of comments containing wording associated with refusal, pulling up,
#   being tailed off, or finishing a long way behind.
#
# No value is decoded or converted in this cell.

large_numeric_btn_values = pd.read_sql_query(
    f"""
    SELECT
        CAST(btn AS REAL) AS raw_numeric_btn,

        COUNT(*) AS runner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        COUNT(DISTINCT course) AS courses,

        MIN(CAST(ovr_btn AS REAL)) AS min_numeric_ovr_btn,

        MAX(CAST(ovr_btn AS REAL)) AS max_numeric_ovr_btn,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%refus%'
                THEN 1 ELSE 0
            END
        ) AS refusal_wording_rows,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%tailed off%'
                THEN 1 ELSE 0
            END
        ) AS tailed_off_wording_rows,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%pulled up%'
                THEN 1 ELSE 0
            END
        ) AS pulled_up_wording_rows,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%no chance%'
                  OR LOWER(comment) LIKE '%well behind%'
                  OR LOWER(comment) LIKE '%detached%'
                THEN 1 ELSE 0
            END
        ) AS heavily_beaten_wording_rows

    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(btn) IN ('integer', 'real')
      AND CAST(btn AS REAL) > 50

    GROUP BY CAST(btn AS REAL)

    ORDER BY
        runner_records DESC,
        raw_numeric_btn
    """,
    connection,
)

print(
    "Distinct numeric btn values above 50: "
    f"{len(large_numeric_btn_values):,}"
)
print(
    "Runner records with numeric btn above 50: "
    f"{large_numeric_btn_values['runner_records'].sum():,}"
)

large_numeric_btn_values

Distinct numeric btn values above 50: 100
Runner records with numeric btn above 50: 8,818


,raw_numeric_btn,runner_records,provisional_races,courses,min_numeric_ovr_btn,max_numeric_ovr_btn,refusal_wording_rows,tailed_off_wording_rows,pulled_up_wording_rows,heavily_beaten_wording_rows
0,99.0,689,680,95,99.00,288.00,9,426,72,77
1,51.0,464,462,99,51.00,264.25,0,293,1,60
2,54.0,394,393,91,54.00,228.25,0,256,13,56
3,53.0,389,387,96,53.00,219.75,0,248,11,54
4,52.0,387,387,96,52.00,199.75,0,237,12,47
...,...,...,...,...,...,...,...,...,...,...
95,148.0,1,1,1,128.75,128.75,0,1,1,0
96,149.0,1,1,1,115.00,115.00,0,1,0,0
97,151.0,1,1,1,167.75,167.75,0,1,0,0
98,182.0,1,1,1,43.75,43.75,1,0,0,0


In [32]:
# Test whether unusually large btn values may contain concatenated fractional
# racing margins rather than literal numeric distances.
#
# Candidate interpretation for values ending in:
# - 1: whole number + 1/4
# - 2: whole number + 1/2
# - 3: whole number + 3/4
#
# Examples:
# - 121 -> 12.25
# - 122 -> 12.50
# - 123 -> 12.75
# - 182 -> 18.50
#
# The test compares two arithmetic possibilities:
# 1. the raw numeric btn value;
# 2. the candidate decoded fractional value.
#
# No raw value is changed and no mapping is accepted yet.

fractional_btn_encoding_test = pd.read_sql_query(
    f"""
    WITH numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS raw_numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,
            horse,
            comment,

            LAG(CAST(ovr_btn AS REAL)) OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS previous_ovr_btn,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(btn) IN ('integer', 'real')
          AND typeof(ovr_btn) IN ('integer', 'real')
    ),
    candidate_values AS (
        SELECT
            *,

            -- Decode a final digit of 1, 2, or 3 as a possible quarter,
            -- half, or three-quarter suffix.
            CASE
                WHEN raw_numeric_btn >= 100
                 AND CAST(raw_numeric_btn AS INTEGER) % 10 = 1
                THEN
                    CAST(CAST(raw_numeric_btn AS INTEGER) / 10 AS INTEGER)
                    + 0.25

                WHEN raw_numeric_btn >= 100
                 AND CAST(raw_numeric_btn AS INTEGER) % 10 = 2
                THEN
                    CAST(CAST(raw_numeric_btn AS INTEGER) / 10 AS INTEGER)
                    + 0.50

                WHEN raw_numeric_btn >= 100
                 AND CAST(raw_numeric_btn AS INTEGER) % 10 = 3
                THEN
                    CAST(CAST(raw_numeric_btn AS INTEGER) / 10 AS INTEGER)
                    + 0.75
            END AS decoded_fractional_btn

        FROM numeric_finishers
        WHERE numeric_finish_order > 1
          AND raw_numeric_btn >= 100
          AND CAST(raw_numeric_btn AS INTEGER) % 10 IN (1, 2, 3)
    )
    SELECT
        CASE
            WHEN decoded_fractional_btn IS NULL
            THEN 'not_decodable'

            WHEN ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + decoded_fractional_btn)
            ) < 0.000001
            THEN 'decoded_exact_match'

            WHEN ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + decoded_fractional_btn)
            ) <= 0.25
            THEN 'decoded_within_0.25'

            WHEN ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + decoded_fractional_btn)
            )
            <
            ABS(
                numeric_ovr_btn
                - (previous_ovr_btn + raw_numeric_btn)
            )
            THEN 'decoded_closer_but_above_0.25'

            ELSE 'raw_value_as_close_or_closer'
        END AS decoding_result,

        COUNT(*) AS runner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        MIN(raw_numeric_btn) AS min_raw_btn,
        MAX(raw_numeric_btn) AS max_raw_btn

    FROM candidate_values
    GROUP BY decoding_result
    ORDER BY runner_records DESC
    """,
    connection,
)

fractional_btn_encoding_test

,decoding_result,runner_records,provisional_races,min_raw_btn,max_raw_btn
0,raw_value_as_close_or_closer,40,40,101.0,153.0
1,decoded_closer_but_above_0.25,1,1,182.0,182.0


In [33]:
# Profile races where the first positive numeric position has a non-zero
# cumulative beaten-distance value.
#
# These may arise when:
# - the original winner was disqualified and is represented separately;
# - the lowest surviving numeric position is not position 1;
# - the race result is incomplete or amended;
# - the source distance sequence is anomalous.
#
# This cell summarises the surrounding result structure only. It does not
# assume that the lowest numeric position is necessarily the official winner.

first_numeric_finisher_nonzero_profile = pd.read_sql_query(
    f"""
    WITH ranked_numeric_finishers AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_id,
            race_name,
            ran,
            CAST(pos AS INTEGER) AS numeric_pos,
            CAST(btn AS REAL) AS numeric_btn,
            CAST(ovr_btn AS REAL) AS numeric_ovr_btn,
            horse,
            comment,

            ROW_NUMBER() OVER (
                PARTITION BY date, course, off
                ORDER BY
                    CAST(pos AS INTEGER),
                    rowid
            ) AS numeric_finish_order

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
          AND typeof(ovr_btn) IN ('integer', 'real')
    ),
    affected_races AS (
        SELECT
            date,
            course,
            off,
            numeric_pos AS first_numeric_pos,
            numeric_btn AS first_numeric_btn,
            numeric_ovr_btn AS first_numeric_ovr_btn
        FROM ranked_numeric_finishers
        WHERE numeric_finish_order = 1
          AND ABS(numeric_ovr_btn) >= 0.000001
    ),
    race_context AS (
        SELECT
            d.date,
            d.course,
            d.off,

            MAX(
                CASE
                    WHEN typeof(d.pos) = 'text'
                     AND TRIM(CAST(d.pos AS TEXT)) = 'DSQ'
                    THEN 1 ELSE 0
                END
            ) AS contains_dsq,

            MAX(
                CASE
                    WHEN typeof(d.pos) = 'integer'
                     AND CAST(d.pos AS INTEGER) = 0
                    THEN 1 ELSE 0
                END
            ) AS contains_zero_pos,

            MIN(
                CASE
                    WHEN typeof(d.pos) = 'integer'
                     AND CAST(d.pos AS INTEGER) > 0
                    THEN CAST(d.pos AS INTEGER)
                END
            ) AS lowest_positive_numeric_pos,

            COUNT(*) AS source_rows,
            MIN(d.ran) AS recorded_ran

        FROM {SOURCE_TABLE} AS d
        INNER JOIN affected_races AS a
            ON d.date = a.date
           AND d.course = a.course
           AND d.off = a.off
        WHERE d.rowid <> 1
        GROUP BY
            d.date,
            d.course,
            d.off
    )
    SELECT
        CASE
            WHEN race_context.contains_dsq = 1
            THEN 'race_contains_dsq'

            WHEN race_context.contains_zero_pos = 1
            THEN 'race_contains_pos_zero'

            WHEN race_context.lowest_positive_numeric_pos > 1
            THEN 'numeric_positions_start_above_1'

            WHEN race_context.source_rows < race_context.recorded_ran
            THEN 'source_rows_below_ran'

            ELSE 'other_nonzero_first_distance'
        END AS context_pattern,

        COUNT(*) AS provisional_races,

        MIN(affected_races.first_numeric_pos) AS min_first_numeric_pos,
        MAX(affected_races.first_numeric_pos) AS max_first_numeric_pos,

        MIN(affected_races.first_numeric_ovr_btn) AS min_first_ovr_btn,
        MAX(affected_races.first_numeric_ovr_btn) AS max_first_ovr_btn

    FROM affected_races
    INNER JOIN race_context
        ON affected_races.date = race_context.date
       AND affected_races.course = race_context.course
       AND affected_races.off = race_context.off

    GROUP BY context_pattern
    ORDER BY provisional_races DESC
    """,
    connection,
)

first_numeric_finisher_nonzero_profile

,context_pattern,provisional_races,min_first_numeric_pos,max_first_numeric_pos,min_first_ovr_btn,max_first_ovr_btn
0,other_nonzero_first_distance,323,1,1,0.05,4.50
1,race_contains_dsq,176,1,1,0.05,30.25


In [34]:
# Profile the 323 races where:
# - the first positive numeric position is position 1;
# - its ovr_btn value is non-zero;
# - the race contains no DSQ or pos = 0;
# - the source-row count is not below ran.
#
# The cell groups these races by the winner's raw btn and ovr_btn values and
# searches comments for explicit amendment language.
#
# This remains descriptive profiling. No winner distance is reset to zero and
# no amended-result classification is assigned automatically.

unexplained_nonzero_winner_distance_profile = pd.read_sql_query(
    f"""
    WITH race_context AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran,

            MAX(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
                    THEN 1 ELSE 0
                END
            ) AS contains_dsq,

            MAX(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) = 0
                    THEN 1 ELSE 0
                END
            ) AS contains_zero_pos

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    numeric_winners AS (
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,
            d.race_id,
            d.race_name,
            d.ran,
            d.num,
            CAST(d.btn AS REAL) AS winner_btn,
            CAST(d.ovr_btn AS REAL) AS winner_ovr_btn,
            d.horse,
            d.comment
        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS context
            ON d.date = context.date
           AND d.course = context.course
           AND d.off = context.off
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
          AND CAST(d.pos AS INTEGER) = 1
          AND typeof(d.ovr_btn) IN ('integer', 'real')
          AND ABS(CAST(d.ovr_btn AS REAL)) >= 0.000001
          AND context.contains_dsq = 0
          AND context.contains_zero_pos = 0
          AND context.source_rows >= context.recorded_ran
    )
    SELECT
        winner_btn,
        winner_ovr_btn,

        COUNT(*) AS winner_records,

        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,

        COUNT(DISTINCT course) AS courses,

        SUM(
            CASE
                WHEN LOWER(comment) LIKE '%promoted%'
                  OR LOWER(comment) LIKE '%placed first%'
                  OR LOWER(comment) LIKE '%placed 1st%'
                  OR LOWER(comment) LIKE '%awarded%'
                  OR LOWER(comment) LIKE '%finished second%'
                  OR LOWER(comment) LIKE '%finished 2nd%'
                  OR LOWER(comment) LIKE '%finished third%'
                  OR LOWER(comment) LIKE '%finished 3rd%'
                THEN 1
                ELSE 0
            END
        ) AS explicit_amendment_wording_rows,

        SUM(
            CASE
                WHEN TRIM(comment) = ''
                THEN 1
                ELSE 0
            END
        ) AS blank_comment_rows

    FROM numeric_winners
    GROUP BY
        winner_btn,
        winner_ovr_btn
    ORDER BY
        winner_records DESC,
        winner_ovr_btn,
        winner_btn
    """,
    connection,
)

print(
    "Distinct winner btn/ovr_btn patterns: "
    f"{len(unexplained_nonzero_winner_distance_profile):,}"
)
print(
    "Winner records represented: "
    f"{unexplained_nonzero_winner_distance_profile['winner_records'].sum():,}"
)

unexplained_nonzero_winner_distance_profile

Distinct winner btn/ovr_btn patterns: 20
Winner records represented: 324


,winner_btn,winner_ovr_btn,winner_records,provisional_races,courses,explicit_amendment_wording_rows,blank_comment_rows
0,0.05,0.05,81,81,57,79,2
1,0.20,0.20,73,73,53,72,1
2,0.10,0.10,65,65,46,63,2
3,0.30,0.30,36,36,31,33,3
4,0.50,0.50,22,22,20,21,1
5,0.75,0.75,12,12,12,11,1
6,1.00,1.00,5,5,5,5,0
7,1.25,1.25,5,5,5,5,0
8,1.75,1.75,5,5,5,5,0
9,2.00,2.00,5,5,5,2,2


In [35]:
# Retrieve:
# 1. the non-zero-distance race containing more than one recorded winner;
# 2. non-zero-distance winners without explicit amendment wording.
#
# These are the residual cases needed before concluding how amended winners
# are represented.
#
# Raw values remain unchanged and no candidate correction is applied.

nonzero_winner_distance_exceptions = pd.read_sql_query(
    f"""
    WITH race_context AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran,

            MAX(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
                    THEN 1 ELSE 0
                END
            ) AS contains_dsq,

            MAX(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) = 0
                    THEN 1 ELSE 0
                END
            ) AS contains_zero_pos

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    affected_winners AS (
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,
            d.race_id,
            d.race_name,
            d.ran,
            d.num,
            d.pos,
            d.btn,
            d.ovr_btn,
            d.horse,
            d.jockey,
            d.trainer,
            d.comment,

            CASE
                WHEN LOWER(d.comment) LIKE '%promoted%'
                  OR LOWER(d.comment) LIKE '%placed first%'
                  OR LOWER(d.comment) LIKE '%placed 1st%'
                  OR LOWER(d.comment) LIKE '%awarded%'
                  OR LOWER(d.comment) LIKE '%finished second%'
                  OR LOWER(d.comment) LIKE '%finished 2nd%'
                  OR LOWER(d.comment) LIKE '%finished third%'
                  OR LOWER(d.comment) LIKE '%finished 3rd%'
                THEN 1
                ELSE 0
            END AS explicit_amendment_wording

        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS context
            ON d.date = context.date
           AND d.course = context.course
           AND d.off = context.off
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
          AND CAST(d.pos AS INTEGER) = 1
          AND typeof(d.ovr_btn) IN ('integer', 'real')
          AND ABS(CAST(d.ovr_btn AS REAL)) >= 0.000001
          AND context.contains_dsq = 0
          AND context.contains_zero_pos = 0
          AND context.source_rows >= context.recorded_ran
    ),
    winner_counts AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS recorded_winners
        FROM affected_winners
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        winners.*,
        counts.recorded_winners,

        CASE
            WHEN counts.recorded_winners > 1
            THEN 'multiple_recorded_winners'
            ELSE 'no_explicit_amendment_wording'
        END AS exception_reason

    FROM affected_winners AS winners
    INNER JOIN winner_counts AS counts
        ON winners.date = counts.date
       AND winners.course = counts.course
       AND winners.off = counts.off

    WHERE counts.recorded_winners > 1
       OR winners.explicit_amendment_wording = 0

    ORDER BY
        exception_reason,
        winners.date,
        winners.course,
        winners.off,
        winners.source_rowid
    """,
    connection,
)

print(f"Exception records returned: {len(nonzero_winner_distance_exceptions):,}")

nonzero_winner_distance_exceptions

Exception records returned: 16


,source_rowid,date,course,off,race_id,race_name,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment,explicit_amendment_wording,recorded_winners,exception_reason
0,1530125,2024-06-29,San Isidro (ARG),10:40,871811,Gran Premio Estrellas Mile (3yo+) (Round) (Turf),16,3,1,2.00,2.00,Bronx (ARG),Cristian E Velazquez,Juan M Etchechoury,Finished dead-heat 2nd - placed dead-heat 1st,0,2,multiple_recorded_winners
1,1530126,2024-06-29,San Isidro (ARG),10:40,871811,Gran Premio Estrellas Mile (3yo+) (Round) (Turf),16,7,1,0.00,2.00,Folie Ninja (ARG),Brian R Enrique,Maria Fernanda Alvarez,Finished dead-heat 2nd - placed dead-heat 1st,0,2,multiple_recorded_winners
2,354325,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),16,8,1,0.20,0.20,Landjunge (GER),Marc Lerner,Waldemar Hickst,,0,1,no_explicit_amendment_wording
3,621406,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & ...,9,10,1,0.30,0.30,Vexatious (USA),Florent Geroux,Neil Drysdale,,0,1,no_explicit_amendment_wording
4,791730,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (...,12,2,1,2.00,2.00,Cheermeister (USA),Reylu Gutierrez,Armando de la Cerda,,0,1,no_explicit_amendment_wording
5,854844,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (D...,7,6,1,0.30,0.30,Congrats This (USA),Miguel Angel Vasquez,Michael Yates,,0,1,no_explicit_amendment_wording
6,855119,2020-04-11,Gulfstream Park (USA),8:52,755716,Maiden Special Weight (Maiden) (3yo+) (Main Tr...,10,7,1,2.00,2.00,Colonel Liam (USA),Luis Saez,Todd Pletcher,,0,1,no_explicit_amendment_wording
7,858575,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (D...,9,2,1,0.30,0.30,Flying Lindy (USA),Jose Angel Medina,J Alan Williams,,0,1,no_explicit_amendment_wording
8,859130,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Tra...,8,5,1,0.50,0.50,Our Anabelle (USA),Jake L Olesiak,Jason Wise,,0,1,no_explicit_amendment_wording
9,1075280,2021-09-18,Laurel Park (USA),9:18,794079,Frank J De Francis Memorial Dash Stakes (3yo+...,6,2,1,0.75,0.75,Jalen Journey (USA),Fergal Lynch,Steven Asmussen,,0,1,no_explicit_amendment_wording


In [36]:
# Retrieve:
# 1. the non-zero-distance race containing more than one recorded winner;
# 2. non-zero-distance winners without explicit amendment wording.
#
# These are the residual cases needed before concluding how amended winners
# are represented.
#
# Raw values remain unchanged and no candidate correction is applied.

nonzero_winner_distance_exceptions = pd.read_sql_query(
    f"""
    WITH race_context AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran,

            MAX(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
                    THEN 1 ELSE 0
                END
            ) AS contains_dsq,

            MAX(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) = 0
                    THEN 1 ELSE 0
                END
            ) AS contains_zero_pos

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    affected_winners AS (
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,
            d.race_id,
            d.race_name,
            d.ran,
            d.num,
            d.pos,
            d.btn,
            d.ovr_btn,
            d.horse,
            d.jockey,
            d.trainer,
            d.comment,

            CASE
                WHEN LOWER(d.comment) LIKE '%promoted%'
                  OR LOWER(d.comment) LIKE '%placed first%'
                  OR LOWER(d.comment) LIKE '%placed 1st%'
                  OR LOWER(d.comment) LIKE '%awarded%'
                  OR LOWER(d.comment) LIKE '%finished second%'
                  OR LOWER(d.comment) LIKE '%finished 2nd%'
                  OR LOWER(d.comment) LIKE '%finished third%'
                  OR LOWER(d.comment) LIKE '%finished 3rd%'
                THEN 1
                ELSE 0
            END AS explicit_amendment_wording

        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS context
            ON d.date = context.date
           AND d.course = context.course
           AND d.off = context.off
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
          AND CAST(d.pos AS INTEGER) = 1
          AND typeof(d.ovr_btn) IN ('integer', 'real')
          AND ABS(CAST(d.ovr_btn AS REAL)) >= 0.000001
          AND context.contains_dsq = 0
          AND context.contains_zero_pos = 0
          AND context.source_rows >= context.recorded_ran
    ),
    winner_counts AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS recorded_winners
        FROM affected_winners
        GROUP BY
            date,
            course,
            off
    )
    SELECT
        winners.*,
        counts.recorded_winners,

        CASE
            WHEN counts.recorded_winners > 1
            THEN 'multiple_recorded_winners'
            ELSE 'no_explicit_amendment_wording'
        END AS exception_reason

    FROM affected_winners AS winners
    INNER JOIN winner_counts AS counts
        ON winners.date = counts.date
       AND winners.course = counts.course
       AND winners.off = counts.off

    WHERE counts.recorded_winners > 1
       OR winners.explicit_amendment_wording = 0

    ORDER BY
        exception_reason,
        winners.date,
        winners.course,
        winners.off,
        winners.source_rowid
    """,
    connection,
)

print(f"Exception records returned: {len(nonzero_winner_distance_exceptions):,}")

nonzero_winner_distance_exceptions

Exception records returned: 16


,source_rowid,date,course,off,race_id,race_name,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment,explicit_amendment_wording,recorded_winners,exception_reason
0,1530125,2024-06-29,San Isidro (ARG),10:40,871811,Gran Premio Estrellas Mile (3yo+) (Round) (Turf),16,3,1,2.00,2.00,Bronx (ARG),Cristian E Velazquez,Juan M Etchechoury,Finished dead-heat 2nd - placed dead-heat 1st,0,2,multiple_recorded_winners
1,1530126,2024-06-29,San Isidro (ARG),10:40,871811,Gran Premio Estrellas Mile (3yo+) (Round) (Turf),16,7,1,0.00,2.00,Folie Ninja (ARG),Brian R Enrique,Maria Fernanda Alvarez,Finished dead-heat 2nd - placed dead-heat 1st,0,2,multiple_recorded_winners
2,354325,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),16,8,1,0.20,0.20,Landjunge (GER),Marc Lerner,Waldemar Hickst,,0,1,no_explicit_amendment_wording
3,621406,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & ...,9,10,1,0.30,0.30,Vexatious (USA),Florent Geroux,Neil Drysdale,,0,1,no_explicit_amendment_wording
4,791730,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (...,12,2,1,2.00,2.00,Cheermeister (USA),Reylu Gutierrez,Armando de la Cerda,,0,1,no_explicit_amendment_wording
5,854844,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (D...,7,6,1,0.30,0.30,Congrats This (USA),Miguel Angel Vasquez,Michael Yates,,0,1,no_explicit_amendment_wording
6,855119,2020-04-11,Gulfstream Park (USA),8:52,755716,Maiden Special Weight (Maiden) (3yo+) (Main Tr...,10,7,1,2.00,2.00,Colonel Liam (USA),Luis Saez,Todd Pletcher,,0,1,no_explicit_amendment_wording
7,858575,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (D...,9,2,1,0.30,0.30,Flying Lindy (USA),Jose Angel Medina,J Alan Williams,,0,1,no_explicit_amendment_wording
8,859130,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Tra...,8,5,1,0.50,0.50,Our Anabelle (USA),Jake L Olesiak,Jason Wise,,0,1,no_explicit_amendment_wording
9,1075280,2021-09-18,Laurel Park (USA),9:18,794079,Frank J De Francis Memorial Dash Stakes (3yo+...,6,2,1,0.75,0.75,Jalen Journey (USA),Fergal Lynch,Steven Asmussen,,0,1,no_explicit_amendment_wording


In [37]:
# Retrieve every source runner record from races containing either:
# - multiple non-zero-distance winners; or
# - a non-zero-distance winner without explicit amendment wording.
#
# Full race context is needed to inspect:
# - the complete position sequence;
# - duplicated or missing positions;
# - cumulative-distance ordering;
# - comments attached to other runners that may explain an amendment.
#
# No raw result value is changed and no candidate correction is applied.

nonzero_winner_exception_races = pd.read_sql_query(
    f"""
    WITH race_context AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran,

            MAX(
                CASE
                    WHEN typeof(pos) = 'text'
                     AND TRIM(CAST(pos AS TEXT)) = 'DSQ'
                    THEN 1 ELSE 0
                END
            ) AS contains_dsq,

            MAX(
                CASE
                    WHEN typeof(pos) = 'integer'
                     AND CAST(pos AS INTEGER) = 0
                    THEN 1 ELSE 0
                END
            ) AS contains_zero_pos

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    affected_winners AS (
        SELECT
            d.date,
            d.course,
            d.off,

            CASE
                WHEN LOWER(d.comment) LIKE '%promoted%'
                  OR LOWER(d.comment) LIKE '%placed first%'
                  OR LOWER(d.comment) LIKE '%placed 1st%'
                  OR LOWER(d.comment) LIKE '%placed dead-heat 1st%'
                  OR LOWER(d.comment) LIKE '%awarded%'
                  OR LOWER(d.comment) LIKE '%finished second%'
                  OR LOWER(d.comment) LIKE '%finished 2nd%'
                  OR LOWER(d.comment) LIKE '%finished third%'
                  OR LOWER(d.comment) LIKE '%finished 3rd%'
                THEN 1
                ELSE 0
            END AS explicit_amendment_wording

        FROM {SOURCE_TABLE} AS d
        INNER JOIN race_context AS context
            ON d.date = context.date
           AND d.course = context.course
           AND d.off = context.off
        WHERE d.rowid <> 1
          AND typeof(d.pos) = 'integer'
          AND CAST(d.pos AS INTEGER) = 1
          AND typeof(d.ovr_btn) IN ('integer', 'real')
          AND ABS(CAST(d.ovr_btn AS REAL)) >= 0.000001
          AND context.contains_dsq = 0
          AND context.contains_zero_pos = 0
          AND context.source_rows >= context.recorded_ran
    ),
    affected_races AS (
        SELECT
            date,
            course,
            off
        FROM affected_winners
        GROUP BY
            date,
            course,
            off
        HAVING COUNT(*) > 1
            OR MAX(explicit_amendment_wording) = 0
    )
    SELECT
        d.rowid AS source_rowid,
        d.date,
        d.course,
        d.race_id,
        d.off,
        d.race_name,
        d.type,
        d.ran,
        d.num,
        d.pos,
        d.btn,
        d.ovr_btn,
        d.horse,
        d.jockey,
        d.trainer,
        d.comment
    FROM {SOURCE_TABLE} AS d
    INNER JOIN affected_races AS a
        ON d.date = a.date
       AND d.course = a.course
       AND d.off = a.off
    WHERE d.rowid <> 1
    ORDER BY
        d.date,
        d.course,
        d.off,
        CASE
            WHEN typeof(d.pos) = 'integer'
            THEN CAST(d.pos AS INTEGER)
            ELSE 999
        END,
        d.rowid
    """,
    connection,
)

print(
    "Affected provisional races: "
    f"{nonzero_winner_exception_races[['date', 'course', 'off']].drop_duplicates().shape[0]:,}"
)
print(f"Runner records returned: {len(nonzero_winner_exception_races):,}")

nonzero_winner_exception_races

Affected provisional races: 15
Runner records returned: 150


,source_rowid,date,course,race_id,off,race_name,type,ran,num,pos,btn,ovr_btn,horse,jockey,trainer,comment
0,354325,2017-04-11,Saint-Cloud (FR),673268,12:47,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,16,8,1,0.2,0.2,Landjunge (GER),Marc Lerner,Waldemar Hickst,
1,354324,2017-04-11,Saint-Cloud (FR),673268,12:47,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,16,4,2,0.75,1,Max La Fripouille (FR),Theo Bachelot,Rod Collet,
2,354326,2017-04-11,Saint-Cloud (FR),673268,12:47,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,16,13,3,0,0,Nardo (FR),Ronan Thomas,F Foucher,
3,354323,2017-04-11,Saint-Cloud (FR),673268,12:47,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,16,3,4,0.75,1.75,Eagle Eyes (GER),Mickael Barzalona,Jean-Pierre Carvalho,
4,354322,2017-04-11,Saint-Cloud (FR),673268,12:47,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,16,5,5,0.3,2,Astronomy Alfred (FR),Alexis Badel,Stephen Ramsay,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,1836914,2026-05-01,Palermo,919641,19:05,Gran Premio Montevideo (Dirt),Flat,7,7,3,0,0,Colorado Del Monte (ARG),Gustavo Emiliano Calvente,Nicolas Martin Ferro,
146,1836917,2026-05-01,Palermo,919641,19:05,Gran Premio Montevideo (Dirt),Flat,7,6,4,1,1.5,Cardo Castilla (ARG),William Pereyra,Juan F Saldivia,
147,1836918,2026-05-01,Palermo,919641,19:05,Gran Premio Montevideo (Dirt),Flat,7,5,5,0.2,1.5,Desert Voice (ARG),Juan C Noriega,Juan F Saldivia,
148,1836919,2026-05-01,Palermo,919641,19:05,Gran Premio Montevideo (Dirt),Flat,7,2,6,12,13.5,Thomas Shelby (ARG),Kevin Banegas,Gonzalo Sebastian Peralta,


In [38]:
# Summarise the 15 races containing unexplained non-zero winner distances.
#
# The full 150-row display is too large for dependable visual inspection, so
# this cell produces one row per provisional race and tests:
# - whether numeric positions are duplicated;
# - whether any ordinal positions are missing;
# - whether ovr_btn decreases as position increases;
# - whether any later runner returns to ovr_btn = 0;
# - whether comments anywhere in the race contain amendment wording.
#
# This remains descriptive profiling. No result or distance is corrected.

nonzero_winner_exception_summary = (
    nonzero_winner_exception_races
    .assign(
        # Convert position and distance fields to nullable numeric helpers.
        numeric_pos=pd.to_numeric(
            nonzero_winner_exception_races["pos"],
            errors="coerce",
        ).astype("Int64"),
        numeric_ovr_btn=pd.to_numeric(
            nonzero_winner_exception_races["ovr_btn"],
            errors="coerce",
        ),

        # Search all runner comments, not only the winner's comment, for wording
        # that may explain an amended result.
        amendment_wording=(
            nonzero_winner_exception_races["comment"]
            .fillna("")
            .str.lower()
            .str.contains(
                r"disqual|placed|promoted|awarded|demoted|finished dead-heat",
                regex=True,
            )
        ),
    )
    .sort_values(
        ["date", "course", "off", "numeric_pos", "source_rowid"]
    )
)

# Compare each cumulative distance with the previous runner in the displayed
# numeric-position order.
nonzero_winner_exception_summary["previous_ovr_btn"] = (
    nonzero_winner_exception_summary
    .groupby(["date", "course", "off"])["numeric_ovr_btn"]
    .shift(1)
)

nonzero_winner_exception_summary["ovr_btn_decreases"] = (
    nonzero_winner_exception_summary["numeric_ovr_btn"]
    < nonzero_winner_exception_summary["previous_ovr_btn"]
)

race_level_nonzero_winner_summary = (
    nonzero_winner_exception_summary
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "ran",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        # Count the source rows and distinct numeric positions.
        source_rows=("source_rowid", "count"),
        distinct_numeric_positions=("numeric_pos", "nunique"),

        # Record the observed numeric-position range.
        min_numeric_pos=("numeric_pos", "min"),
        max_numeric_pos=("numeric_pos", "max"),

        # Count duplicated numeric positions.
        duplicated_position_rows=(
            "numeric_pos",
            lambda values: values.duplicated(keep=False).sum(),
        ),

        # Count cumulative-distance reversals.
        ovr_btn_decrease_rows=("ovr_btn_decreases", "sum"),

        # Count runners after first place whose cumulative distance is zero.
        later_zero_ovr_btn_rows=(
            "numeric_ovr_btn",
            lambda values: (
                values.iloc[1:].fillna(float("inf")).eq(0).sum()
            ),
        ),

        # Record whether any comment in the race explains an amendment.
        amendment_wording_rows=("amendment_wording", "sum"),

        # Preserve the ordered position and cumulative-distance sequences.
        observed_positions=(
            "numeric_pos",
            lambda values: ", ".join(
                str(int(value)) for value in values.dropna()
            ),
        ),
        observed_ovr_btn=(
            "numeric_ovr_btn",
            lambda values: ", ".join(
                f"{value:g}" for value in values.dropna()
            ),
        ),
    )
)

race_level_nonzero_winner_summary

,date,course,off,race_id,race_name,ran,source_rows,distinct_numeric_positions,min_numeric_pos,max_numeric_pos,duplicated_position_rows,ovr_btn_decrease_rows,later_zero_ovr_btn_rows,amendment_wording_rows,observed_positions,observed_ovr_btn
0,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),16,16,16,1,16,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,...","0.2, 1, 0, 1.75, 2, 2.75, 3, 4, 4.75, 5, 5.25,..."
1,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & ...,9,9,9,1,9,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9","0.3, 1.5, 0, 1.75, 2.5, 11.25, 12.75, 19.75, 24.5"
2,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (...,12,12,12,1,12,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12","2, 0, 3.75, 4.75, 5, 8, 9.25, 11.25, 12.75, 14..."
3,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (D...,7,7,7,1,7,0,1,1,0,"1, 2, 3, 4, 5, 6, 7","0.3, 0, 5.5, 9, 11.5, 14.25, 19.75"
4,2020-04-11,Gulfstream Park (USA),8:52,755716,Maiden Special Weight (Maiden) (3yo+) (Main Tr...,10,10,9,1,9,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9","2, 0, 5, 6.5, 8, 12, 13.5, 15.75, 16"
5,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (D...,9,9,9,1,9,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9","0.3, 0, 1.5, 4.25, 8.75, 16, 20.25, 20.5, 46.5"
6,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Tra...,8,8,8,1,8,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8","0.5, 0, 1.75, 2.25, 2.5, 5, 8.25, 10.5"
7,2021-09-18,Laurel Park (USA),9:18,794079,Frank J De Francis Memorial Dash Stakes (3yo+...,6,6,6,1,6,0,1,1,1,"1, 2, 3, 4, 5, 6","0.75, 0, 2.5, 7.25, 8.75, 9.25"
8,2023-08-31,Longchamp (FR),12:48,848966,Prix de Fontenoy (Maiden) (Unraced 2yo Colts &...,8,8,8,1,8,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8","1.5, 0, 1.75, 5.75, 7.75, 7.75, 9.25, 13.75"
9,2023-09-08,Saint-Cloud (FR),1:35,849331,Prix Jean-Claude Desaint (Championnat Paris-Tu...,12,12,12,1,12,0,1,1,0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12","0.1, 0, 1.25, 2, 4, 4.5, 6.25, 6.5, 9, 13, 13...."


In [39]:
# Identify where the minimum cumulative beaten distance occurs in each of the
# 15 residual amended or inconsistent races.
#
# In a conventional unamended result, the minimum ovr_btn would normally occur
# at official position 1. In amended results, it may remain attached to the
# horse that crossed the line first but was subsequently placed lower.
#
# This cell records:
# - the minimum observed ovr_btn;
# - every official position sharing that minimum;
# - whether the minimum occurs at position 1;
# - whether the minimum is exactly zero.
#
# No original finishing position is inferred and no source value is altered.

minimum_distance_position_summary = (
    nonzero_winner_exception_summary
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "ran",
        ],
        as_index=False,
        dropna=False,
    )
    .apply(
        lambda race: pd.Series(
            {
                # Record the lowest cumulative beaten-distance value observed
                # anywhere in the provisional race.
                "minimum_ovr_btn": race["numeric_ovr_btn"].min(),

                # Preserve every official position assigned that minimum value.
                "positions_at_minimum_ovr_btn": ", ".join(
                    str(int(position))
                    for position in race.loc[
                        race["numeric_ovr_btn"].eq(
                            race["numeric_ovr_btn"].min()
                        ),
                        "numeric_pos",
                    ].dropna()
                ),

                # Count how many runner records share the minimum distance.
                "runners_at_minimum_ovr_btn": race[
                    "numeric_ovr_btn"
                ].eq(
                    race["numeric_ovr_btn"].min()
                ).sum(),

                # Test whether the official winner is among the runners sharing
                # the minimum cumulative distance.
                "official_winner_at_minimum": (
                    race.loc[
                        race["numeric_ovr_btn"].eq(
                            race["numeric_ovr_btn"].min()
                        ),
                        "numeric_pos",
                    ]
                    .eq(1)
                    .any()
                ),

                # Test whether the minimum cumulative distance is zero.
                "minimum_is_zero": abs(
                    race["numeric_ovr_btn"].min()
                ) < 0.000001,
            }
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)

minimum_distance_position_summary

,date,course,off,race_id,race_name,ran,minimum_ovr_btn,positions_at_minimum_ovr_btn,runners_at_minimum_ovr_btn,official_winner_at_minimum,minimum_is_zero
0,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),16,0.0,3,1,False,True
1,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & ...,9,0.0,3,1,False,True
2,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (...,12,0.0,2,1,False,True
3,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (D...,7,0.0,2,1,False,True
4,2020-04-11,Gulfstream Park (USA),8:52,755716,Maiden Special Weight (Maiden) (3yo+) (Main Tr...,10,0.0,2,1,False,True
5,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (D...,9,0.0,2,1,False,True
6,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Tra...,8,0.0,2,1,False,True
7,2021-09-18,Laurel Park (USA),9:18,794079,Frank J De Francis Memorial Dash Stakes (3yo+...,6,0.0,2,1,False,True
8,2023-08-31,Longchamp (FR),12:48,848966,Prix de Fontenoy (Maiden) (Unraced 2yo Colts &...,8,0.0,2,1,False,True
9,2023-09-08,Saint-Cloud (FR),1:35,849331,Prix Jean-Claude Desaint (Championnat Paris-Tu...,12,0.0,2,1,False,True


In [41]:
# Summarise the two provisional races containing numeric pos = 0.
#
# Earlier profiling showed:
# - five zero-position records at Nakayama;
# - three zero-position records at Klampenborg.
#
# This cell first identifies the affected provisional race keys, then retrieves
# every runner from those races for a compact race-level summary.
#
# No meaning is assigned to pos = 0 and no replacement outcome is inferred.

# Build a proper MultiIndex containing the provisional race keys where at least
# one source runner record has numeric pos = 0.
zero_position_race_keys = pd.MultiIndex.from_frame(
    affected_races_profile.loc[
        affected_races_profile["zero_pos"].fillna(False),
        ["date", "course", "off"],
    ].drop_duplicates()
)

# Select every runner record belonging to one of those affected races.
zero_position_races = affected_races_profile.loc[
    pd.MultiIndex.from_frame(
        affected_races_profile[["date", "course", "off"]]
    ).isin(zero_position_race_keys)
].copy()

# Add helper fields needed only for readable inspection.
zero_position_races = zero_position_races.assign(
    # Identify blank comments separately from blank runner numbers.
    blank_comment=(
        zero_position_races["comment"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    ),

    # Preserve the complete raw runner record in a compact display string.
    runner_summary=zero_position_races.apply(
        lambda row: (
            f"rowid={row['source_rowid']}; "
            f"num={str(row['num']).strip() or '<blank>'}; "
            f"pos={row['pos']}; "
            f"btn={row['btn']}; "
            f"ovr_btn={row['ovr_btn']}; "
            f"horse={row['horse']}; "
            f"comment="
            f"{'<blank>' if str(row['comment']).strip() == '' else row['comment']}"
        ),
        axis=1,
    ),
)

# Produce one compact summary row per affected provisional race.
zero_position_race_summary = (
    zero_position_races
    .groupby(
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "ran",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        # Count the complete source representation.
        source_rows=("source_rowid", "count"),

        # Count numeric zero-position records.
        zero_pos_rows=("zero_pos", "sum"),

        # Count blank runner numbers and comments.
        blank_num_rows=("blank_num", "sum"),
        blank_comment_rows=("blank_comment", "sum"),

        # Record the highest positive numeric position in the race.
        highest_positive_pos=(
            "numeric_pos",
            lambda values: values[values.gt(0)].max(),
        ),

        # Preserve the raw position sequence.
        observed_positions=(
            "pos_text",
            lambda values: ", ".join(values.tolist()),
        ),

        # Preserve one detailed line per runner for direct inspection.
        runner_records=(
            "runner_summary",
            lambda values: "\n".join(values.tolist()),
        ),
    )
)

zero_position_race_summary

,date,course,off,race_id,race_name,ran,source_rows,zero_pos_rows,blank_num_rows,blank_comment_rows,highest_positive_pos,observed_positions,runner_records
0,2015-04-18,Nakayama (JPN),7:40,624701,Nakayama Grand Jump (Turf),15,15,5,15,15,10,"0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10",rowid=40679; num=<blank>; pos=0; btn=0.0; ovr_...
1,2015-06-28,Klampenborg (DEN),3:00,630668,Dansk Hesteforsikring Scandinavian Open Champi...,13,13,3,0,13,10,"0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10",rowid=77097; num=1; pos=0; btn=0.0; ovr_btn=0....


In [42]:
# Inspect representative source comments for each textual position code.
#
# The compact codes resemble standard racing outcomes, but the notebook should
# validate their source usage before proposing candidate result categories.
#
# Up to five nonblank comments are returned for each code. Codes whose comments
# are usually blank will simply have fewer examples.
#
# Exact raw position codes and comments are preserved unchanged.

text_pos_comment_examples = pd.read_sql_query(
    f"""
    WITH coded_rows AS (
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            race_name,
            type,
            TRIM(CAST(pos AS TEXT)) AS raw_pos,
            horse,
            comment,

            ROW_NUMBER() OVER (
                PARTITION BY TRIM(CAST(pos AS TEXT))
                ORDER BY
                    CASE
                        WHEN TRIM(comment) <> '' THEN 0
                        ELSE 1
                    END,
                    date,
                    course,
                    off,
                    rowid
            ) AS example_number

        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'text'
    )
    SELECT
        raw_pos,
        example_number,
        source_rowid,
        date,
        course,
        off,
        race_name,
        type,
        horse,
        comment
    FROM coded_rows
    WHERE example_number <= 5
    ORDER BY
        raw_pos,
        example_number
    """,
    connection,
)

pd.set_option("display.max_colwidth", 180)

text_pos_comment_examples

,raw_pos,example_number,source_rowid,date,course,off,race_name,type,horse,comment
0,BD,1,5802,2015-01-18,Leopardstown (IRE),2:50,BoyleSports Hurdle (Extended Handicap Hurdle - Rated 0-150),Hurdle,Clondaw Warrior (IRE),Held up - towards rear when brought down 4 out(op 8/1)
1,BD,2,5803,2015-01-18,Leopardstown (IRE),2:50,BoyleSports Hurdle (Extended Handicap Hurdle - Rated 0-150),Hurdle,Cacheofgold (IRE),Never better than mid-division - towards rear when brought down 4 out
2,BD,3,5980,2015-01-19,Exeter,1:50,Cheltenham Preview Evening 3rd March Maiden Hurdle (Div II),Hurdle,Greybougg (GB),Mid-division - pushed along when brought down 4th
3,BD,4,6081,2015-01-20,Wetherby,3:15,Download New Racing UK iPad App National Hunt Novices Hurdle,Hurdle,Ozzy Thomas (IRE),Tracked leaders - brought down 3rd(op 2/1)
4,BD,5,6158,2015-01-21,Ayr,1:15,Back of the Net At BetVictor.com Mares Maiden Hurdle,Hurdle,Just For Pleasure (IRE),Chased clear leaders - took closer order 7th - 2 lengths down and chasing leader when brought down 3 out(op 4/1 tchd 5/1)
5,CO,1,880,2015-01-02,Ffos Las,1:10,Radio Carmarthenshire and Scarlet FM Handicap Chase,Chase,Victory Gunner (IRE),In touch in 5th - headway to chase leaders 6th - ridden 4 out - staying on and disputing close 2nd when carried out last
6,CO,2,6189,2015-01-21,Ayr,1:50,BetVictor.com National Hunt Maiden Hurdle,Hurdle,Simply Lucky (IRE),Chased leader until left in lead 4th - carried out by loose horse next
7,CO,3,23664,2015-03-11,Cheltenham,4:00,Glenfarclas Handicap Chase (A Cross Country Chase),Chase,Quantitativeeasing (IRE),Held up in midfield - progress from 4 out - poised to challenge going strongly when carried out after 2 out
8,CO,4,37227,2015-04-11,Aintree,5:10,One Stop Energy Handicap Hurdle (For Conditional Jockeys and Amateur Riders),Hurdle,Cinders And Ashes (GB),Held up in rear on inner - carried out just after 2nd
9,CO,5,40206,2015-04-17,Tipperary (IRE),4:40,Join The Tipperary Races Supporters Club Maiden Hurdle,Hurdle,Misdflight (IRE),Mid-division - 9th after 1st and raced keenly - carried out before next


In [43]:
# Define a notebook-level candidate mapping for the validated textual
# finishing-outcome codes.
#
# The mapping is grounded in the observed source comments and remains separate
# from the immutable raw pos value.
#
# This is a candidate semantic layer only:
# - raw pos remains unchanged;
# - no final schema is designed;
# - numeric positions and pos = 0 are handled separately;
# - DSQ remains distinct because it can retain numeric distance values.

candidate_text_result_mapping = pd.DataFrame(
    [
        {
            "raw_pos": "BD",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "brought_down",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "CO",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "carried_out",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "DSQ",
            "candidate_result_category": "disqualified",
            "candidate_outcome": "disqualified",
            "completed_course": pd.NA,
            "started_race": True,
            "distance_expected_numeric": True,
        },
        {
            "raw_pos": "F",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "fell",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "LFT",
            "candidate_result_category": "did_not_participate_fully",
            "candidate_outcome": "left_or_failed_to_take_part",
            "completed_course": False,
            "started_race": pd.NA,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "PU",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "pulled_up",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "REF",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "refused_at_obstacle",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "RO",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "ran_out",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "RR",
            "candidate_result_category": "did_not_start",
            "candidate_outcome": "refused_to_race",
            "completed_course": False,
            "started_race": False,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "SU",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "slipped_up",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
        {
            "raw_pos": "UR",
            "candidate_result_category": "non_finisher",
            "candidate_outcome": "unseated_rider",
            "completed_course": False,
            "started_race": True,
            "distance_expected_numeric": False,
        },
    ]
).convert_dtypes()

candidate_text_result_mapping

,raw_pos,candidate_result_category,candidate_outcome,completed_course,started_race,distance_expected_numeric
0,BD,non_finisher,brought_down,False,True,False
1,CO,non_finisher,carried_out,False,True,False
2,DSQ,disqualified,disqualified,<NA>,True,True
3,F,non_finisher,fell,False,True,False
4,LFT,did_not_participate_fully,left_or_failed_to_take_part,False,<NA>,False
5,PU,non_finisher,pulled_up,False,True,False
6,REF,non_finisher,refused_at_obstacle,False,True,False
7,RO,non_finisher,ran_out,False,True,False
8,RR,did_not_start,refused_to_race,False,False,False
9,SU,non_finisher,slipped_up,False,True,False


In [44]:
# Validate the candidate textual-result mapping against the complete source.
#
# The checks confirm that:
# - every observed textual pos code has exactly one candidate mapping;
# - the mapping contains no unused codes;
# - joining the mapping to the source neither loses nor duplicates rows;
# - the mapped row counts reproduce the previously observed code frequencies.
#
# This validates mapping coverage only. It does not yet apply candidate
# attributes to numeric positions or pos = 0.

observed_text_pos_counts = pd.read_sql_query(
    f"""
    SELECT
        TRIM(CAST(pos AS TEXT)) AS raw_pos,
        COUNT(*) AS source_rows
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) = 'text'
    GROUP BY TRIM(CAST(pos AS TEXT))
    ORDER BY raw_pos
    """,
    connection,
)

# Join the complete observed code inventory to the candidate mapping.
text_mapping_validation = observed_text_pos_counts.merge(
    candidate_text_result_mapping,
    on="raw_pos",
    how="outer",
    indicator=True,
    validate="one_to_one",
)

# Add clear validation flags without changing the mapping itself.
text_mapping_validation["observed_code_is_mapped"] = (
    text_mapping_validation["_merge"].eq("both")
)

text_mapping_validation["mapping_code_is_observed"] = (
    text_mapping_validation["_merge"].eq("both")
)

print(
    "Observed textual pos codes: "
    f"{observed_text_pos_counts['raw_pos'].nunique():,}"
)
print(
    "Candidate mapping rows: "
    f"{candidate_text_result_mapping['raw_pos'].nunique():,}"
)
print(
    "Observed textual source rows: "
    f"{observed_text_pos_counts['source_rows'].sum():,}"
)
print(
    "Unmapped observed codes: "
    f"{text_mapping_validation['_merge'].eq('left_only').sum():,}"
)
print(
    "Unused mapping codes: "
    f"{text_mapping_validation['_merge'].eq('right_only').sum():,}"
)

text_mapping_validation.drop(columns="_merge")

Observed textual pos codes: 11
Candidate mapping rows: 11
Observed textual source rows: 94,611
Unmapped observed codes: 0
Unused mapping codes: 0


,raw_pos,source_rows,candidate_result_category,candidate_outcome,completed_course,started_race,distance_expected_numeric,observed_code_is_mapped,mapping_code_is_observed
0,BD,1020,non_finisher,brought_down,False,True,False,True,True
1,CO,77,non_finisher,carried_out,False,True,False,True,True
2,DSQ,619,disqualified,disqualified,<NA>,True,True,True,True
3,F,15681,non_finisher,fell,False,True,False,True,True
4,LFT,7,did_not_participate_fully,left_or_failed_to_take_part,False,<NA>,False,True,True
5,PU,65832,non_finisher,pulled_up,False,True,False,True,True
6,REF,291,non_finisher,refused_at_obstacle,False,True,False,True,True
7,RO,463,non_finisher,ran_out,False,True,False,True,True
8,RR,723,did_not_start,refused_to_race,False,False,False,True,True
9,SU,371,non_finisher,slipped_up,False,True,False,True,True


In [45]:
# Classify every source runner record into a broad candidate result
# representation using only the source patterns validated so far.
#
# This is not a final target schema. It is a notebook-level profiling layer
# designed to establish whether the complete source can be represented without
# hiding raw anomalies.
#
# Rules:
# - positive integer pos -> numeric_finishing_position;
# - integer pos = 0 -> unresolved_zero_position;
# - text pos = DSQ -> disqualified;
# - every other text pos -> mapped textual outcome from the validated table.
#
# Raw pos, btn, and ovr_btn values remain unchanged.

# Load only the result fields needed for complete-source classification.
source_result_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        pos,
        ran,
        btn,
        ovr_btn
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

# Preserve the exact raw position as text for joining and display.
source_result_rows["raw_pos_text"] = (
    source_result_rows["pos"]
    .astype(str)
    .str.strip()
)

# Identify SQLite/Python position representation before deriving any candidate
# semantic attributes.
source_result_rows["pos_is_numeric"] = pd.to_numeric(
    source_result_rows["pos"],
    errors="coerce",
).notna()

source_result_rows["numeric_pos"] = pd.to_numeric(
    source_result_rows["pos"],
    errors="coerce",
).astype("Int64")

# Join validated candidate meanings onto textual position codes only.
source_result_profile = source_result_rows.merge(
    candidate_text_result_mapping[
        [
            "raw_pos",
            "candidate_result_category",
            "candidate_outcome",
            "completed_course",
            "started_race",
            "distance_expected_numeric",
        ]
    ],
    left_on="raw_pos_text",
    right_on="raw_pos",
    how="left",
    validate="many_to_one",
)

# Derive one broad candidate representation for every source row.
source_result_profile["candidate_result_representation"] = pd.NA

source_result_profile.loc[
    source_result_profile["numeric_pos"].gt(0).fillna(False),
    "candidate_result_representation",
] = "numeric_finishing_position"

source_result_profile.loc[
    source_result_profile["numeric_pos"].eq(0).fillna(False),
    "candidate_result_representation",
] = "unresolved_zero_position"

source_result_profile.loc[
    source_result_profile["raw_pos_text"].eq("DSQ"),
    "candidate_result_representation",
] = "disqualified"

source_result_profile.loc[
    source_result_profile["raw_pos"].notna()
    & source_result_profile["raw_pos_text"].ne("DSQ"),
    "candidate_result_representation",
] = "mapped_textual_outcome"

# Summarise full-source coverage and distance representation.
candidate_result_representation_summary = (
    source_result_profile
    .groupby(
        [
            "candidate_result_representation",
            "candidate_result_category",
            "candidate_outcome",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_records=("source_rowid", "count"),
        min_numeric_pos=("numeric_pos", "min"),
        max_numeric_pos=("numeric_pos", "max"),
        numeric_btn_rows=(
            "btn",
            lambda values: pd.to_numeric(values, errors="coerce").notna().sum(),
        ),
        numeric_ovr_btn_rows=(
            "ovr_btn",
            lambda values: pd.to_numeric(values, errors="coerce").notna().sum(),
        ),
    )
    .sort_values(
        ["candidate_result_representation", "runner_records"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print(f"Source runner records classified: {len(source_result_profile):,}")
print(
    "Unclassified source runner records: "
    f"{source_result_profile['candidate_result_representation'].isna().sum():,}"
)

candidate_result_representation_summary

Source runner records classified: 1,851,285
Unclassified source runner records: 0


,candidate_result_representation,candidate_result_category,candidate_outcome,runner_records,min_numeric_pos,max_numeric_pos,numeric_btn_rows,numeric_ovr_btn_rows
0,disqualified,disqualified,disqualified,619,<NA>,<NA>,619,619
1,mapped_textual_outcome,non_finisher,pulled_up,65832,<NA>,<NA>,0,0
2,mapped_textual_outcome,non_finisher,fell,15681,<NA>,<NA>,0,0
3,mapped_textual_outcome,non_finisher,unseated_rider,9527,<NA>,<NA>,0,0
4,mapped_textual_outcome,non_finisher,brought_down,1020,<NA>,<NA>,0,0
5,mapped_textual_outcome,did_not_start,refused_to_race,723,<NA>,<NA>,0,0
6,mapped_textual_outcome,non_finisher,ran_out,463,<NA>,<NA>,0,0
7,mapped_textual_outcome,non_finisher,slipped_up,371,<NA>,<NA>,0,0
8,mapped_textual_outcome,non_finisher,refused_at_obstacle,291,<NA>,<NA>,0,0
9,mapped_textual_outcome,non_finisher,carried_out,77,<NA>,<NA>,0,0


In [46]:
# Add candidate validation flags for the result anomalies established so far.
#
# These flags remain separate from the candidate result representation:
# - they describe source quality or unusual result structure;
# - they do not overwrite pos, ran, btn, or ovr_btn;
# - one source row may legitimately carry more than one flag.
#
# Flags covered here:
# - unresolved numeric pos = 0;
# - numeric pos greater than ran;
# - duplicate positive numeric position within a provisional race;
# - duplicate position with inconsistent ovr_btn values;
# - race source-row count below ran;
# - externally verified Morphettville source anomaly.

result_validation_flags = pd.read_sql_query(
    f"""
    WITH race_counts AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    duplicate_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runners_at_position,
            COUNT(
                DISTINCT CAST(ovr_btn AS TEXT)
            ) AS distinct_ovr_btn_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
        HAVING COUNT(*) > 1
    )
    SELECT
        d.rowid AS source_rowid,

        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) = 0
            THEN 1 ELSE 0
        END AS unresolved_zero_position_flag,

        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) > d.ran
            THEN 1 ELSE 0
        END AS position_above_ran_flag,

        CASE
            WHEN duplicates.runners_at_position > 1
            THEN 1 ELSE 0
        END AS duplicate_numeric_position_flag,

        CASE
            WHEN duplicates.runners_at_position > 1
             AND duplicates.distinct_ovr_btn_values > 1
            THEN 1 ELSE 0
        END AS duplicate_position_distance_conflict_flag,

        CASE
            WHEN counts.source_rows < counts.recorded_ran
            THEN 1 ELSE 0
        END AS race_rows_below_ran_flag,

        CASE
            WHEN d.rowid = 55516
            THEN 1 ELSE 0
        END AS externally_verified_source_anomaly_flag

    FROM {SOURCE_TABLE} AS d

    INNER JOIN race_counts AS counts
        ON d.date = counts.date
       AND d.course = counts.course
       AND d.off = counts.off

    LEFT JOIN duplicate_positions AS duplicates
        ON d.date = duplicates.date
       AND d.course = duplicates.course
       AND d.off = duplicates.off
       AND typeof(d.pos) = 'integer'
       AND CAST(d.pos AS INTEGER) = duplicates.numeric_pos

    WHERE d.rowid <> 1
    """,
    connection,
)

# Produce one row per observed combination of validation flags.
result_validation_flag_summary = (
    result_validation_flags
    .groupby(
        [
            "unresolved_zero_position_flag",
            "position_above_ran_flag",
            "duplicate_numeric_position_flag",
            "duplicate_position_distance_conflict_flag",
            "race_rows_below_ran_flag",
            "externally_verified_source_anomaly_flag",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_records=("source_rowid", "count"),
    )
    .sort_values(
        "runner_records",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "Runner records carrying at least one validation flag: "
    f"{result_validation_flags.drop(columns='source_rowid').any(axis=1).sum():,}"
)

result_validation_flag_summary

Runner records carrying at least one validation flag: 6,072


,unresolved_zero_position_flag,position_above_ran_flag,duplicate_numeric_position_flag,duplicate_position_distance_conflict_flag,race_rows_below_ran_flag,externally_verified_source_anomaly_flag,runner_records
0,0,0,0,0,0,0,1845213
1,0,0,1,0,0,0,6020
2,0,0,0,0,1,0,32
3,0,1,0,0,0,0,10
4,1,0,0,0,0,0,8
5,0,0,1,1,0,1,1
6,0,0,1,1,0,0,1


In [48]:
# Recalculate the refined result summary directly from the complete source.
#
# The previous race-position group count incorrectly joined to
# affected_races_profile, which contains only the small anomaly-race subset.
# This version counts supported dead-heat groups from the full source query.
#
# Ordinary supported dead heats remain separate from validation issues.

refined_result_flag_summary = pd.read_sql_query(
    f"""
    WITH race_counts AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    duplicate_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runners_at_position,
            COUNT(
                DISTINCT CAST(ovr_btn AS TEXT)
            ) AS distinct_ovr_btn_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
        HAVING COUNT(*) > 1
    ),
    row_flags AS (
        SELECT
            d.rowid AS source_rowid,
            d.date,
            d.course,
            d.off,

            CASE
                WHEN duplicates.runners_at_position > 1
                 AND duplicates.distinct_ovr_btn_values = 1
                THEN 1 ELSE 0
            END AS candidate_dead_heat_flag,

            CASE
                WHEN duplicates.runners_at_position > 1
                 AND duplicates.distinct_ovr_btn_values > 1
                THEN 1 ELSE 0
            END AS duplicate_position_distance_conflict_flag,

            CASE
                WHEN typeof(d.pos) = 'integer'
                 AND CAST(d.pos AS INTEGER) = 0
                THEN 1 ELSE 0
            END AS unresolved_zero_position_flag,

            CASE
                WHEN typeof(d.pos) = 'integer'
                 AND CAST(d.pos AS INTEGER) > d.ran
                THEN 1 ELSE 0
            END AS position_above_ran_flag,

            CASE
                WHEN counts.source_rows < counts.recorded_ran
                THEN 1 ELSE 0
            END AS incomplete_race_context_flag,

            CASE
                WHEN d.rowid = 55516
                THEN 1 ELSE 0
            END AS externally_verified_source_anomaly_flag

        FROM {SOURCE_TABLE} AS d

        INNER JOIN race_counts AS counts
            ON d.date = counts.date
           AND d.course = counts.course
           AND d.off = counts.off

        LEFT JOIN duplicate_positions AS duplicates
            ON d.date = duplicates.date
           AND d.course = duplicates.course
           AND d.off = duplicates.off
           AND typeof(d.pos) = 'integer'
           AND CAST(d.pos AS INTEGER) = duplicates.numeric_pos

        WHERE d.rowid <> 1
    )
    SELECT
        'candidate dead-heat runner records' AS measure,
        SUM(candidate_dead_heat_flag) AS count
    FROM row_flags

    UNION ALL

    SELECT
        'candidate dead-heat race-position groups',
        COUNT(*)
    FROM duplicate_positions
    WHERE distinct_ovr_btn_values = 1

    UNION ALL

    SELECT
        'runner records in validation-issue context',
        SUM(
            CASE
                WHEN duplicate_position_distance_conflict_flag = 1
                  OR unresolved_zero_position_flag = 1
                  OR position_above_ran_flag = 1
                  OR incomplete_race_context_flag = 1
                  OR externally_verified_source_anomaly_flag = 1
                THEN 1
                ELSE 0
            END
        )
    FROM row_flags

    UNION ALL

    SELECT
        'distinct source rows with runner-level anomaly',
        SUM(
            CASE
                WHEN duplicate_position_distance_conflict_flag = 1
                  OR unresolved_zero_position_flag = 1
                  OR position_above_ran_flag = 1
                  OR externally_verified_source_anomaly_flag = 1
                THEN 1
                ELSE 0
            END
        )
    FROM row_flags

    UNION ALL

    SELECT
        'incomplete provisional races',
        COUNT(*)
    FROM race_counts
    WHERE source_rows < recorded_ran

    UNION ALL

    SELECT
        'duplicate-position distance-conflict rows',
        SUM(duplicate_position_distance_conflict_flag)
    FROM row_flags

    UNION ALL

    SELECT
        'externally verified anomaly rows',
        SUM(externally_verified_source_anomaly_flag)
    FROM row_flags
    """,
    connection,
)

refined_result_flag_summary

,measure,count
0,candidate dead-heat runner records,6020
1,candidate dead-heat race-position groups,3006
2,runner records in validation-issue context,52
3,distinct source rows with runner-level anomaly,20
4,incomplete provisional races,5
5,duplicate-position distance-conflict rows,2
6,externally verified anomaly rows,1


In [49]:
# Assemble a notebook-level candidate runner-result view from the validated
# source conventions and anomaly checks.
#
# This is deliberately not a final target schema. It demonstrates which
# candidate attributes can be derived reproducibly while retaining:
# - exact raw result fields;
# - source row lineage;
# - supported dead-heat evidence;
# - unresolved and contradictory source patterns.
#
# Candidate attributes:
# - numeric finishing position, where raw pos is a positive integer;
# - raw textual outcome code and validated semantic mapping;
# - broad result representation;
# - candidate dead-heat flag;
# - distance availability;
# - separate validation flags.
#
# No raw source value is corrected or overwritten.

candidate_runner_results = pd.read_sql_query(
    f"""
    WITH race_counts AS (
        SELECT
            date,
            course,
            off,
            COUNT(*) AS source_rows,
            MIN(ran) AS recorded_ran
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off
    ),
    duplicate_positions AS (
        SELECT
            date,
            course,
            off,
            CAST(pos AS INTEGER) AS numeric_pos,
            COUNT(*) AS runners_at_position,
            COUNT(
                DISTINCT CAST(ovr_btn AS TEXT)
            ) AS distinct_ovr_btn_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND typeof(pos) = 'integer'
          AND CAST(pos AS INTEGER) > 0
        GROUP BY
            date,
            course,
            off,
            CAST(pos AS INTEGER)
        HAVING COUNT(*) > 1
    )
    SELECT
        -- Physical source lineage.
        d.rowid AS source_rowid,

        -- Current provisional race identity.
        d.date,
        d.course,
        d.off,

        -- Preserved source descriptors and result fields.
        d.race_id AS raw_race_id,
        d.ran AS raw_ran,
        d.num AS raw_num,
        d.pos AS raw_pos,
        d.btn AS raw_btn,
        d.ovr_btn AS raw_ovr_btn,
        d.horse,

        -- Candidate numeric finishing position.
        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) > 0
            THEN CAST(d.pos AS INTEGER)
        END AS candidate_finish_position,

        -- Preserve textual outcome codes separately from numeric positions.
        CASE
            WHEN typeof(d.pos) = 'text'
            THEN TRIM(CAST(d.pos AS TEXT))
        END AS raw_outcome_code,

        -- Broad source-result representation.
        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) > 0
            THEN 'numeric_finishing_position'

            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) = 0
            THEN 'unresolved_zero_position'

            WHEN typeof(d.pos) = 'text'
             AND TRIM(CAST(d.pos AS TEXT)) = 'DSQ'
            THEN 'disqualified'

            WHEN typeof(d.pos) = 'text'
            THEN 'mapped_textual_outcome'
        END AS candidate_result_representation,

        -- Candidate dead heat requires both duplicated positive position and
        -- shared cumulative beaten distance.
        CASE
            WHEN duplicates.runners_at_position > 1
             AND duplicates.distinct_ovr_btn_values = 1
            THEN 1 ELSE 0
        END AS candidate_dead_heat_flag,

        -- Record whether both beaten-distance fields are numeric.
        CASE
            WHEN typeof(d.btn) IN ('integer', 'real')
             AND typeof(d.ovr_btn) IN ('integer', 'real')
            THEN 1 ELSE 0
        END AS numeric_distance_available_flag,

        -- Validation flags remain independent of the candidate result.
        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) = 0
            THEN 1 ELSE 0
        END AS unresolved_zero_position_flag,

        CASE
            WHEN typeof(d.pos) = 'integer'
             AND CAST(d.pos AS INTEGER) > d.ran
            THEN 1 ELSE 0
        END AS position_above_ran_flag,

        CASE
            WHEN duplicates.runners_at_position > 1
             AND duplicates.distinct_ovr_btn_values > 1
            THEN 1 ELSE 0
        END AS duplicate_position_distance_conflict_flag,

        CASE
            WHEN counts.source_rows < counts.recorded_ran
            THEN 1 ELSE 0
        END AS incomplete_race_context_flag,

        CASE
            WHEN d.rowid = 55516
            THEN 1 ELSE 0
        END AS externally_verified_source_anomaly_flag

    FROM {SOURCE_TABLE} AS d

    INNER JOIN race_counts AS counts
        ON d.date = counts.date
       AND d.course = counts.course
       AND d.off = counts.off

    LEFT JOIN duplicate_positions AS duplicates
        ON d.date = duplicates.date
       AND d.course = duplicates.course
       AND d.off = duplicates.off
       AND typeof(d.pos) = 'integer'
       AND CAST(d.pos AS INTEGER) = duplicates.numeric_pos

    WHERE d.rowid <> 1
    """,
    connection,
)

# Join the validated semantic mapping for textual outcome codes.
candidate_runner_results = candidate_runner_results.merge(
    candidate_text_result_mapping[
        [
            "raw_pos",
            "candidate_result_category",
            "candidate_outcome",
            "completed_course",
            "started_race",
            "distance_expected_numeric",
        ]
    ],
    left_on="raw_outcome_code",
    right_on="raw_pos",
    how="left",
    validate="many_to_one",
)

# Summarise complete-source coverage of the assembled candidate attributes.
candidate_runner_result_summary = (
    candidate_runner_results
    .groupby(
        [
            "candidate_result_representation",
            "candidate_result_category",
            "candidate_outcome",
            "candidate_dead_heat_flag",
            "numeric_distance_available_flag",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        runner_records=("source_rowid", "count"),
        validation_issue_rows=(
            "source_rowid",
            lambda values: candidate_runner_results.loc[
                values.index,
                [
                    "unresolved_zero_position_flag",
                    "position_above_ran_flag",
                    "duplicate_position_distance_conflict_flag",
                    "incomplete_race_context_flag",
                    "externally_verified_source_anomaly_flag",
                ],
            ]
            .any(axis=1)
            .sum(),
        ),
    )
    .sort_values(
        ["candidate_result_representation", "runner_records"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print(f"Candidate runner-result rows: {len(candidate_runner_results):,}")
print(
    "Rows without a candidate result representation: "
    f"{candidate_runner_results['candidate_result_representation'].isna().sum():,}"
)

candidate_runner_result_summary

Candidate runner-result rows: 1,851,285
Rows without a candidate result representation: 0


,candidate_result_representation,candidate_result_category,candidate_outcome,candidate_dead_heat_flag,numeric_distance_available_flag,runner_records,validation_issue_rows
0,disqualified,disqualified,disqualified,0,1,619,0
1,mapped_textual_outcome,non_finisher,pulled_up,0,0,65832,0
2,mapped_textual_outcome,non_finisher,fell,0,0,15681,0
3,mapped_textual_outcome,non_finisher,unseated_rider,0,0,9527,0
4,mapped_textual_outcome,non_finisher,brought_down,0,0,1020,0
5,mapped_textual_outcome,did_not_start,refused_to_race,0,0,723,0
6,mapped_textual_outcome,non_finisher,ran_out,0,0,463,0
7,mapped_textual_outcome,non_finisher,slipped_up,0,0,371,0
8,mapped_textual_outcome,non_finisher,refused_at_obstacle,0,0,291,0
9,mapped_textual_outcome,non_finisher,carried_out,0,0,77,0


In [50]:
# Validate the assembled candidate runner-result view against the established
# full-source totals and notebook findings.
#
# These checks confirm:
# - one candidate row exists for every source data row;
# - source_rowid remains unique;
# - the result representations reconcile to the complete source total;
# - candidate dead-heat counts reproduce the validated full-source counts;
# - textual outcome counts reproduce the validated mapping inventory;
# - anomaly and incomplete-race counts remain unchanged.
#
# No source or candidate value is modified.

candidate_result_validation_checks = pd.DataFrame(
    [
        {
            "check": "candidate rows equal source data rows",
            "observed": len(candidate_runner_results),
            "expected": 1_851_285,
        },
        {
            "check": "source_rowid is unique",
            "observed": candidate_runner_results[
                "source_rowid"
            ].nunique(),
            "expected": 1_851_285,
        },
        {
            "check": "unclassified candidate representations",
            "observed": candidate_runner_results[
                "candidate_result_representation"
            ].isna().sum(),
            "expected": 0,
        },
        {
            "check": "positive numeric finishing-position rows",
            "observed": candidate_runner_results[
                "candidate_finish_position"
            ].notna().sum(),
            "expected": 1_756_666,
        },
        {
            "check": "candidate dead-heat runner records",
            "observed": candidate_runner_results[
                "candidate_dead_heat_flag"
            ].sum(),
            "expected": 6_020,
        },
        {
            "check": "textual outcome rows including DSQ",
            "observed": candidate_runner_results[
                "raw_outcome_code"
            ].notna().sum(),
            "expected": 94_611,
        },
        {
            "check": "mapped non-DSQ textual outcome rows",
            "observed": candidate_runner_results[
                "candidate_result_representation"
            ].eq("mapped_textual_outcome").sum(),
            "expected": 93_992,
        },
        {
            "check": "disqualified rows",
            "observed": candidate_runner_results[
                "candidate_result_representation"
            ].eq("disqualified").sum(),
            "expected": 619,
        },
        {
            "check": "unresolved zero-position rows",
            "observed": candidate_runner_results[
                "unresolved_zero_position_flag"
            ].sum(),
            "expected": 8,
        },
        {
            "check": "position-above-ran rows",
            "observed": candidate_runner_results[
                "position_above_ran_flag"
            ].sum(),
            "expected": 10,
        },
        {
            "check": "duplicate-position distance-conflict rows",
            "observed": candidate_runner_results[
                "duplicate_position_distance_conflict_flag"
            ].sum(),
            "expected": 2,
        },
        {
            "check": "rows in incomplete race context",
            "observed": candidate_runner_results[
                "incomplete_race_context_flag"
            ].sum(),
            "expected": 32,
        },
        {
            "check": "externally verified anomaly rows",
            "observed": candidate_runner_results[
                "externally_verified_source_anomaly_flag"
            ].sum(),
            "expected": 1,
        },
    ]
)

candidate_result_validation_checks["passed"] = (
    candidate_result_validation_checks["observed"]
    == candidate_result_validation_checks["expected"]
)

print(
    "Validation checks passed: "
    f"{candidate_result_validation_checks['passed'].sum():,} / "
    f"{len(candidate_result_validation_checks):,}"
)

candidate_result_validation_checks

Validation checks passed: 13 / 13


,check,observed,expected,passed
0,candidate rows equal source data rows,1851285,1851285,True
1,source_rowid is unique,1851285,1851285,True
2,unclassified candidate representations,0,0,True
3,positive numeric finishing-position rows,1756666,1756666,True
4,candidate dead-heat runner records,6020,6020,True
5,textual outcome rows including DSQ,94611,94611,True
6,mapped non-DSQ textual outcome rows,93992,93992,True
7,disqualified rows,619,619,True
8,unresolved zero-position rows,8,8,True
9,position-above-ran rows,10,10,True


In [51]:
# Record the bounded result-attribute decisions supported by Notebook 05.
#
# This is not a final schema specification. It distinguishes:
# - exact raw source attributes that must be preserved;
# - candidate attributes that can be derived reproducibly;
# - validation flags that must remain separate from the result itself;
# - unresolved concepts that must not be normalised automatically.
#
# Each recommendation is grounded in the complete-source profiling and
# reconciliation checks completed above.

result_attribute_decision_register = pd.DataFrame(
    [
        {
            "attribute_area": "Source lineage",
            "attribute": "source_rowid",
            "status": "preserve",
            "reason": (
                "Provides immutable physical lineage to the exact SQLite "
                "source runner record."
            ),
        },
        {
            "attribute_area": "Raw race context",
            "attribute": "raw_ran",
            "status": "preserve",
            "reason": (
                "Matches source-row count in 189,038 of 189,043 races but must "
                "remain independent because five extracts contain fewer rows "
                "than the recorded field size."
            ),
        },
        {
            "attribute_area": "Raw result",
            "attribute": "raw_pos",
            "status": "preserve",
            "reason": (
                "Carries positive numeric positions, numeric zero, and 11 "
                "validated textual outcome codes in one source field."
            ),
        },
        {
            "attribute_area": "Raw distance",
            "attribute": "raw_btn",
            "status": "preserve",
            "reason": (
                "Usually represents an incremental beaten distance but is not "
                "universally additive and includes large jurisdiction- or "
                "source-dependent values."
            ),
        },
        {
            "attribute_area": "Raw distance",
            "attribute": "raw_ovr_btn",
            "status": "preserve",
            "reason": (
                "Usually represents cumulative beaten distance but may remain "
                "anchored to the original on-course order after amended results."
            ),
        },
        {
            "attribute_area": "Candidate result",
            "attribute": "candidate_finish_position",
            "status": "derive",
            "reason": (
                "Reliably derived only where raw pos is a positive integer; "
                "covers 1,756,666 source runner records."
            ),
        },
        {
            "attribute_area": "Candidate result",
            "attribute": "raw_outcome_code",
            "status": "derive_without_replacing_raw",
            "reason": (
                "Separates the 11 validated textual outcome codes from numeric "
                "finishing positions while retaining the exact raw pos value."
            ),
        },
        {
            "attribute_area": "Candidate result",
            "attribute": "candidate_result_representation",
            "status": "derive",
            "reason": (
                "Every source row can be classified as numeric finishing "
                "position, mapped textual outcome, disqualified, or unresolved "
                "zero position."
            ),
        },
        {
            "attribute_area": "Candidate outcome",
            "attribute": "candidate_outcome",
            "status": "derive_for_text_codes",
            "reason": (
                "All 11 textual codes are validated through source comments and "
                "map one-to-one to stable candidate semantic labels."
            ),
        },
        {
            "attribute_area": "Candidate outcome",
            "attribute": "candidate_result_category",
            "status": "derive_for_text_codes",
            "reason": (
                "Provides broad categories such as non-finisher, did not start, "
                "did not participate fully, and disqualified."
            ),
        },
        {
            "attribute_area": "Dead heats",
            "attribute": "candidate_dead_heat_flag",
            "status": "derive_with_condition",
            "reason": (
                "Supported where positive numeric positions are duplicated and "
                "all tied runners share the same raw ovr_btn; identifies 3,006 "
                "race-position groups and 6,020 runner records."
            ),
        },
        {
            "attribute_area": "Distance availability",
            "attribute": "numeric_distance_available_flag",
            "status": "derive",
            "reason": (
                "Numeric distance is consistently available for positive "
                "numeric positions, DSQ, and pos = 0, but absent for ordinary "
                "textual non-finish and non-start outcomes."
            ),
        },
        {
            "attribute_area": "Disqualification",
            "attribute": "disqualified outcome",
            "status": "preserve_as_separate_concept",
            "reason": (
                "All 619 DSQ rows retain numeric distance values, often "
                "describing the original on-course finish before amendment."
            ),
        },
        {
            "attribute_area": "Unresolved result",
            "attribute": "pos = 0",
            "status": "preserve_and_flag",
            "reason": (
                "Occurs on eight runners in two complete races with no comments "
                "or outcome codes; no defensible non-finish meaning can be "
                "inferred."
            ),
        },
        {
            "attribute_area": "Validation",
            "attribute": "position_above_ran_flag",
            "status": "derive",
            "reason": (
                "Identifies ten source rows whose numeric position exceeds ran; "
                "these include amended and contradictory result structures."
            ),
        },
        {
            "attribute_area": "Validation",
            "attribute": "duplicate_position_distance_conflict_flag",
            "status": "derive",
            "reason": (
                "Separates the two Morphettville conflict rows from ordinary "
                "supported dead heats."
            ),
        },
        {
            "attribute_area": "Validation",
            "attribute": "incomplete_race_context_flag",
            "status": "derive_at_race_level",
            "reason": (
                "Five provisional races contain a continuous positive placing "
                "sequence but fewer source rows than recorded ran."
            ),
        },
        {
            "attribute_area": "Verified anomaly",
            "attribute": "externally_verified_source_anomaly",
            "status": "preserve_as_audit_record",
            "reason": (
                "Cinnamon Carter is externally verified as tied 12th rather "
                "than the raw source position 10; raw values must not be "
                "overwritten."
            ),
        },
        {
            "attribute_area": "Deferred design",
            "attribute": "final staging schema",
            "status": "deferred",
            "reason": (
                "Notebook 05 establishes source representation and candidate "
                "attributes only; final table structure and keys remain outside "
                "the bounded study."
            ),
        },
    ]
)

result_attribute_decision_register

,attribute_area,attribute,status,reason
0,Source lineage,source_rowid,preserve,Provides immutable physical lineage to the exact SQLite source runner record.
1,Raw race context,raw_ran,preserve,"Matches source-row count in 189,038 of 189,043 races but must remain independent because five extracts contain fewer rows than the recorded field size."
2,Raw result,raw_pos,preserve,"Carries positive numeric positions, numeric zero, and 11 validated textual outcome codes in one source field."
3,Raw distance,raw_btn,preserve,Usually represents an incremental beaten distance but is not universally additive and includes large jurisdiction- or source-dependent values.
4,Raw distance,raw_ovr_btn,preserve,Usually represents cumulative beaten distance but may remain anchored to the original on-course order after amended results.
5,Candidate result,candidate_finish_position,derive,"Reliably derived only where raw pos is a positive integer; covers 1,756,666 source runner records."
6,Candidate result,raw_outcome_code,derive_without_replacing_raw,Separates the 11 validated textual outcome codes from numeric finishing positions while retaining the exact raw pos value.
7,Candidate result,candidate_result_representation,derive,"Every source row can be classified as numeric finishing position, mapped textual outcome, disqualified, or unresolved zero position."
8,Candidate outcome,candidate_outcome,derive_for_text_codes,All 11 textual codes are validated through source comments and map one-to-one to stable candidate semantic labels.
9,Candidate outcome,candidate_result_category,derive_for_text_codes,"Provides broad categories such as non-finisher, did not start, did not participate fully, and disqualified."


## Conclusion

### Main conclusion

The source result fields can be represented reliably without replacing or normalising the raw values.

A later staging layer can derive a structured result representation from `pos`, while preserving `ran`, `pos`, `btn`, and `ovr_btn` exactly as supplied.

### Supporting evidence

All 1,851,285 source runner records can be classified into one of four candidate representations:

- 1,756,666 positive numeric finishing positions;
- 93,992 mapped textual outcomes;
- 619 disqualified runners;
- 8 unresolved zero-position records.

The 11 textual `pos` codes are stable source conventions validated through the runner comments:

- `BD` — brought down;
- `CO` — carried out;
- `DSQ` — disqualified;
- `F` — fell;
- `LFT` — left or failed to take part;
- `PU` — pulled up;
- `REF` — refused at an obstacle;
- `RO` — ran out;
- `RR` — refused to race;
- `SU` — slipped up;
- `UR` — unseated rider.

Duplicate positive numeric positions normally represent dead heats. There are 3,006 supported duplicated race-position groups covering 6,020 runner records. These groups share the same cumulative beaten-distance value and generally follow the expected skipped-rank sequence.

`DSQ` must remain separate from ordinary non-finish outcomes. All 619 disqualified runners retain numeric `btn` and `ovr_btn` values, often preserving the on-course finish before the official result was amended.

The beaten-distance fields cannot be treated as a universally additive or position-consistent sequence. Small arithmetic differences are widespread, while amended results can leave `ovr_btn` anchored to the original on-course order rather than the final official `pos`.

### Source anomalies and unresolved cases

The source contains a small bounded set of result anomalies:

- 8 runners use numeric `pos = 0` across two complete races;
- 10 runners have numeric `pos > ran`;
- 2 runners share a duplicated position but have conflicting cumulative distances;
- 5 provisional races contain fewer source rows than the recorded `ran`;
- 1 Morphettville source row was externally verified as incorrectly recording Cinnamon Carter in 10th rather than dead-heating for 12th.

These records should be preserved and flagged, not silently repaired.

### Candidate attributes

The source supports deriving:

- a positive numeric finishing position;
- a raw textual outcome code;
- a mapped candidate outcome;
- a broad candidate result category;
- a candidate dead-heat flag;
- a numeric-distance-availability flag;
- separate runner-level and race-level validation flags.

### Design implication

Raw result fields and candidate result attributes must remain separate.

In particular:

- `pos` must not be replaced by a normalised result field;
- `DSQ` must not be converted into a numeric finishing position inferred from distance;
- `btn` and `ovr_btn` must not be recalculated to force arithmetic consistency;
- `pos = 0` must remain unresolved;
- ordinary dead heats must not be treated as data-quality failures;
- incomplete race extracts must be flagged at race level;
- externally verified corrections should be retained as audit records rather than overwriting the source.

Final staging-table and target-schema design remains deferred.